In [1]:
!pip install -q -U trl peft bitsandbytes accelerate transformers

^C


**Generate Dataset**

In [2]:
import json
import random
import uuid
import string
from tqdm import tqdm

# ============================================================
# 1. NAMING STRATEGIES (Prevents Overfitting)
# ============================================================
def generate_mixed_name(base_concept):
    """
    Generates tool names in different styles so the model 
    doesn't overfit to just UUIDs or just English names.
    """
    strategy = random.choice(["clear", "snake", "camel", "enterprise", "uuid", "abstract"])
    
    if strategy == "clear":
        # Simple English: "Calculator", "WeatherService"
        return base_concept.title() + random.choice(["Service", "Tool", "API", "Checker", ""])
        
    elif strategy == "snake":
        # Code style: "math_calculator_v1", "weather_get"
        suffix = random.choice(["_v1", "_api", "_func", "_lib", ""])
        return f"{base_concept.lower()}_{random.choice(['core', 'main', 'util', 'kit'])}{suffix}"
        
    elif strategy == "camel":
        # Java/JS style: "mathCalculator", "weatherGet"
        return base_concept.lower() + random.choice(["Service", "Manager", "Handler", "Utils"])
        
    elif strategy == "enterprise":
        # Java Enterprise style: "AbstractMathProvider", "GlobalWeatherSingleton"
        return f"Global{base_concept.title()}{random.choice(['Provider', 'Interface', 'Bean'])}"
        
    elif strategy == "uuid":
        # Cloud/Microservice style: "svc-84ba20", "func-a1"
        return f"svc-{uuid.uuid4().hex[:6]}"
        
    elif strategy == "abstract":
        # Generic names: "Module_A", "System_12"
        # This FORCES the model to read the Description, because the name means nothing.
        return f"System_{random.choice(string.ascii_uppercase)}{random.randint(1,99)}"

def generate_mixed_arg(base_arg):
    """Randomizes argument names so model relies on schema, not keywords"""
    strategy = random.choice(["clear", "short", "technical"])
    
    if strategy == "clear":
        return base_arg # e.g., "city"
    elif strategy == "short":
        return base_arg[0] # e.g., "c" (Hard mode!)
    elif strategy == "technical":
        return f"{base_arg}_val" # e.g., "city_val"

# ============================================================
# 2. DOMAIN LOGIC
# ============================================================
DOMAINS = {
    "calculator": {
        "concepts": ["Math", "Calc", "Compute", "Number"],
        "desc": [
            "Evaluates mathematical expressions.",
            "Performs arithmetic operations on input strings.",
            "Calculation engine for numeric problems."
        ],
        "args": ["expression", "equation", "formula"],
        "queries": [("Calculate 10 + 10", "10 + 10"), ("Solve 5 * 5", "5 * 5")]
    },
    "weather": {
        "concepts": ["Weather", "Meteo", "Climate", "Sky"],
        "desc": [
            "Returns current weather conditions for a location.",
            "Forecast lookup service.",
            "Checks temperature and precipitation."
        ],
        "args": ["city", "location", "place"],
        "queries": [("Weather in London", "London"), ("Check Paris temp", "Paris")]
    },
    "translator": {
        "concepts": ["Translate", "Lang", "Linguist", "Polyglot"],
        "desc": [
            "Translates text between languages.",
            "Language conversion interface.",
            "Converts source text to target language."
        ],
        "args": ["text", "input_str", "content"], # Lang is separate
        "queries": [("Translate 'Hi' to Spanish", "Hi", "Spanish")]
    },
    "search": {
        "concepts": ["Search", "Find", "Query", "Lookup"],
        "desc": [
            "Searches the web for information.",
            "Retrieves data matching keywords.",
            "Knowledge base lookup."
        ],
        "args": ["query", "keywords", "term"],
        "queries": [("Search for Cats", "Cats"), ("Who is Batman?", "Batman")]
    },
    "database": {
        "concepts": ["DB", "Sql", "Data", "Store"],
        "desc": [
            "Executes SQL queries against the database.",
            "Retrieves rows from data storage.",
            "Database management interface."
        ],
        "args": ["query_str", "sql_cmd", "command"],
        "queries": [("Get users from DB", "SELECT * FROM users"), ("Find order #99", "SELECT * FROM orders WHERE id=99")]
    }
}

# ============================================================
# 3. GENERATION LOOP
# ============================================================
DATASET_SIZE = 2000
OUTPUT_FILE = "orchestrator_10k.json"
dataset = []
keys = list(DOMAINS.keys())

print(f"Generating {DATASET_SIZE} examples with MIXED NAMING STRATEGIES...")

for _ in tqdm(range(DATASET_SIZE)):
    target_key = random.choice(keys)
    domain_data = DOMAINS[target_key]
    
    # 1. Create the Target Tool (The one we want to use)
    target_query_tuple = random.choice(domain_data["queries"])
    user_query = target_query_tuple[0]
    
    # Randomize Name & Arg
    t_name = generate_mixed_name(random.choice(domain_data["concepts"]))
    t_desc = random.choice(domain_data["desc"])
    
    # Schema Construction
    t_args = {}
    execution_args = {}
    required = []
    
    # Handle single arg vs multi arg
    if target_key == "translator":
        # Translator needs 2 args
        arg1 = generate_mixed_arg("text")
        arg2 = generate_mixed_arg("target_lang")
        t_args[arg1] = {"type": "string", "description": "Text to translate"}
        t_args[arg2] = {"type": "string", "description": "ISO Language code"}
        required = [arg1, arg2]
        execution_args = {arg1: target_query_tuple[1], arg2: target_query_tuple[2]}
    else:
        # Others need 1 arg
        raw_arg = random.choice(domain_data["args"])
        arg1 = generate_mixed_arg(raw_arg)
        t_args[arg1] = {"type": "string", "description": "Input value"}
        required = [arg1]
        execution_args = {arg1: target_query_tuple[1]}

    target_tool = {
        "name": t_name,
        "description": t_desc,
        "input_schema": {
            "type": "object",
            "properties": t_args,
            "required": required
        }
    }
    
    # 2. Create Distractors (Random noise)
    distractors = []
    num_distractors = random.randint(2, 4)
    other_keys = [k for k in keys if k != target_key]
    
    for k in random.sample(other_keys, num_distractors):
        d_data = DOMAINS[k]
        d_name = generate_mixed_name(random.choice(d_data["concepts"]))
        d_desc = random.choice(d_data["desc"])
        
        # Simple schema for distractors
        d_arg = generate_mixed_arg(random.choice(d_data["args"]))
        d_schema = {
            "type": "object", 
            "properties": {d_arg: {"type": "string"}},
            "required": [d_arg]
        }
        
        distractors.append({
            "name": d_name,
            "description": d_desc,
            "input_schema": d_schema
        })
    
    # 3. Combine and Shuffle
    all_tools = [target_tool] + distractors
    random.shuffle(all_tools)
    
    # 4. Create Messages
    sys_msg = f"""You are an intelligent orchestrator.
Select the correct tool based on the description and user query.
Use the EXACT tool name and argument keys from the schema provided.

TOOLS CONFIGURATION:
{json.dumps(all_tools, indent=2)}"""

    plan = {
        "plan_type": "single_step",
        "tool_use": {
            "tool_name": t_name,
            "arguments": execution_args
        }
    }
    
    dataset.append({
        "messages": [
            {"role": "system", "content": sys_msg},
            {"role": "user", "content": user_query},
            {"role": "assistant", "content": json.dumps(plan, indent=2)}
        ]
    })

with open(OUTPUT_FILE, "w") as f:
    json.dump(dataset, f, indent=2)

print("✅ Dataset Generated.")

Generating 2000 examples with MIXED NAMING STRATEGIES...


100%|██████████| 2000/2000 [00:00<00:00, 9016.07it/s]

✅ Dataset Generated.


In [3]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig,
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# --- KAGGLE SETUP ---
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["WANDB_DISABLED"] = "true"

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "/kaggle/working/qwen2.5-orchestrator-10k"

def main():
    # 1. LOAD THE 10K DATASET
    print("Loading 10k dataset...")
    # We use the generic 'json' loader from huggingface datasets
    dataset = load_dataset("json", data_files="orchestrator_10k.json", split="train")
    
    # Split Train/Test
    dataset = dataset.train_test_split(test_size=0.05) # 5% for eval is enough for 10k
    print(f"Train size: {len(dataset['train'])} | Test size: {len(dataset['test'])}")

    # 2. Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token

    # 3. Model (4-bit)
    print("Loading Model...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map={"":"cuda:0"},
        trust_remote_code=True
    )
    
    model.gradient_checkpointing_enable()
    model = prepare_model_for_kbit_training(model)

    # 4. LoRA Configuration
    # Slightly higher rank (r=32) for a larger/more complex dataset
    peft_config = LoraConfig(
        r=32, 
        lora_alpha=64, 
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    )

    # 5. Trainer
    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset["train"],
        eval_dataset=dataset["test"],
        peft_config=peft_config,
        processing_class=tokenizer,
        args=SFTConfig(
            output_dir=OUTPUT_DIR,
            per_device_train_batch_size=2,
            gradient_accumulation_steps=8, # Effective batch 16
            learning_rate=2e-4,
            num_train_epochs=1,          # 10k examples is plenty for 1 epoch
            fp16=False,
            logging_steps=50,
            save_strategy="epoch",
            eval_strategy="steps",
            eval_steps=200,
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            dataset_text_field="messages",
            packing=False,
            dataset_kwargs={"add_special_tokens": False}
        )
    )

    print("Starting Training (10k samples)...")
    trainer.train()
    
    print("Saving Adapter...")
    trainer.model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print("Done!")

if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Loading 10k dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Train size: 1900 | Test size: 100


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

...

Truncating eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting Training (10k samples)...


Step,Training Loss,Validation Loss


Saving Adapter...
Done!


In [5]:
ls

orchestrator_10k.json  qwen2.5-orchestrator-10k/  qwen2.5-orchestrator-10k.zip


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [6]:
import shutil
shutil.make_archive('qwen2.5-orchestrator-10k', 'zip', '/kaggle/working/qwen2.5-orchestrator-10k')

'/kaggle/working/qwen2.5-orchestrator-10k.zip'

**Inference **

In [9]:
import os
import json
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig,
)
from peft import PeftModel
from typing import List, Dict, Any, Optional
import re

# --- CONFIGURATION ---
BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH = "/kaggle/working/qwen2.5-orchestrator-10k"  # Path to your saved LoRA adapter
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class ToolOrchestrator:
    def __init__(
        self, 
        base_model_name: str = BASE_MODEL_NAME,
        adapter_path: str = ADAPTER_PATH,
        load_in_4bit: bool = True,
        max_new_tokens: int = 512
    ):
        """
        Initialize the orchestrator with base model + LoRA adapter.
        """
        self.max_new_tokens = max_new_tokens
        self.tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        
        print(f"Loading base model: {base_model_name}")
        
        # Load model with same quantization config as training
        if load_in_4bit and DEVICE == "cuda":
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
            )
            self.model = AutoModelForCausalLM.from_pretrained(
                base_model_name,
                quantization_config=bnb_config,
                device_map="auto",
                trust_remote_code=True,
                torch_dtype=torch.float16
            )
        else:
            self.model = AutoModelForCausalLM.from_pretrained(
                base_model_name,
                torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
                device_map="auto" if DEVICE == "cuda" else None,
                trust_remote_code=True
            )
        
        # Load LoRA adapter
        if os.path.exists(adapter_path):
            print(f"Loading LoRA adapter from: {adapter_path}")
            self.model = PeftModel.from_pretrained(self.model, adapter_path)
            # Don't merge for now - keep adapter separate for flexibility
            # self.model = self.model.merge_and_unload()
        else:
            print(f"Warning: Adapter not found at {adapter_path}, using base model only")
        
        self.model.eval()
        print("Model loaded successfully")

    def format_tools(self, tools: List[Dict[str, Any]]) -> str:
        """Format tools list into the same format as training."""
        return json.dumps(tools, indent=2)

    def build_prompt(
        self, 
        user_query: str, 
        tools: List[Dict[str, Any]], 
        system_prompt: Optional[str] = None
    ) -> str:
        """
        Build the chat prompt matching the training format.
        """
        default_sys = """You are an intelligent orchestrator.
Select the correct tool based on the description and user query.
Use the EXACT tool name and argument keys from the schema provided.

TOOLS CONFIGURATION:"""
        
        sys_content = system_prompt if system_prompt else default_sys
        tools_str = self.format_tools(tools)
        
        # Match training format exactly
        messages = [
            {"role": "system", "content": f"{sys_content}\n{tools_str}"},
            {"role": "user", "content": user_query}
        ]
        
        return self.tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=True
        )

    def extract_json_from_text(self, text: str) -> Optional[str]:
        """Extract JSON block from text, handling markdown and raw formats."""
        # Try markdown code blocks first
        patterns = [
            r'```(?:json)?\s*([\s\S]*?)\s*```',  # ```json ... ``` or ``` ... ```
            r'\{[\s\S]*\}',  # Raw JSON object
        ]
        
        for pattern in patterns:
            matches = re.findall(pattern, text)
            for match in matches:
                try:
                    # Validate it's valid JSON
                    json.loads(match)
                    return match
                except json.JSONDecodeError:
                    continue
        
        # Try to find the outermost JSON object
        try:
            start = text.find('{')
            end = text.rfind('}')
            if start != -1 and end != -1 and end > start:
                candidate = text[start:end+1]
                json.loads(candidate)  # Validate
                return candidate
        except json.JSONDecodeError:
            pass
            
        return None

    def normalize_plan(self, data: Dict[str, Any]) -> Dict[str, Any]:
        """
        Normalize various output formats to standard internal format.
        Handles both training format and alternative formats the model might generate.
        """
        # Debug: print what we received
        print(f"  [Debug] Normalizing format: {list(data.keys())}")
        
        # Format 1: Direct flat format (what we're seeing in error)
        # {"tool_name": "...", "kwargs": {...}}
        if "tool_name" in data and "kwargs" in data:
            return {
                "plan_type": "single_step",
                "tool_use": {
                    "tool_name": data["tool_name"],
                    "arguments": data["kwargs"]
                }
            }
        
        # Format 2: Flat format with 'arguments' instead of 'kwargs'
        # {"tool_name": "...", "arguments": {...}}
        if "tool_name" in data and "arguments" in data:
            return {
                "plan_type": "single_step",
                "tool_use": {
                    "tool_name": data["tool_name"],
                    "arguments": data["arguments"]
                }
            }
        
        # Format 3: Nested format (original training format)
        # {"plan_type": "...", "tool_use": {"tool_name": "...", "arguments": {...}}}
        if "tool_use" in data:
            tool_use = data["tool_use"]
            # Ensure arguments key exists (handle kwargs vs arguments)
            if "kwargs" in tool_use and "arguments" not in tool_use:
                tool_use["arguments"] = tool_use.pop("kwargs")
            return {
                "plan_type": data.get("plan_type", "single_step"),
                "tool_use": tool_use
            }
        
        # Format 4: Just tool selection without args (fallback)
        if "tool_name" in data:
            return {
                "plan_type": "single_step",
                "tool_use": {
                    "tool_name": data["tool_name"],
                    "arguments": data.get("arguments", data.get("kwargs", {}))
                }
            }
            
        return data

    def parse_output(self, generated_text: str) -> Optional[Dict[str, Any]]:
        """
        Parse the model output into structured plan.
        """
        print(f"  [Debug] Raw output preview: {generated_text[:150]}...")
        
        # Extract JSON string
        json_str = self.extract_json_from_text(generated_text)
        if not json_str:
            print(f"  [Debug] No JSON found in output")
            return None
        
        print(f"  [Debug] Extracted JSON: {json_str[:200]}...")
        
        try:
            data = json.loads(json_str)
            normalized = self.normalize_plan(data)
            print(f"  [Debug] Normalized plan: {normalized}")
            return normalized
        except json.JSONDecodeError as e:
            print(f"  [Debug] JSON parse error: {e}")
            return None

    @torch.no_grad()
    def select_tool(
        self, 
        user_query: str, 
        tools: List[Dict[str, Any]], 
        temperature: float = 0.1,
        do_sample: bool = True,
        system_prompt: Optional[str] = None
    ) -> Dict[str, Any]:
        """
        Main inference method: select tool for given query.
        """
        prompt = self.build_prompt(user_query, tools, system_prompt)
        
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        
        outputs = self.model.generate(
            **inputs,
            max_new_tokens=self.max_new_tokens,
            temperature=temperature,
            do_sample=do_sample,
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )
        
        # Decode only the new tokens
        generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
        generated_text = self.tokenizer.decode(generated_tokens, skip_special_tokens=True)
        
        # Parse structured output
        plan = self.parse_output(generated_text)
        
        return {
            "success": plan is not None and "tool_use" in plan,
            "plan": plan,
            "raw_output": generated_text,
            "parsed": plan is not None
        }

    def batch_select(
        self, 
        queries: List[str], 
        tools: List[Dict[str, Any]], 
        **kwargs
    ) -> List[Dict[str, Any]]:
        """Batch inference for multiple queries (same tools)."""
        return [self.select_tool(q, tools, **kwargs) for q in queries]


# --- USAGE EXAMPLES ---

def example_basic_usage():
    """Example: Basic tool selection."""
    orchestrator = ToolOrchestrator(
        adapter_path="/kaggle/working/qwen2.5-orchestrator-10k"
    )
    
    # Define available tools (same format as training)
    available_tools = [
        {
            "name": "math_calculator_v1",
            "description": "Evaluates mathematical expressions.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression to evaluate"}
                },
                "required": ["expression"]
            }
        },
        {
            "name": "weather_get",
            "description": "Returns current weather conditions for a location.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"}
                },
                "required": ["city"]
            }
        },
        {
            "name": "System_A12",
            "description": "Translates text between languages.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "t_val": {"type": "string", "description": "Text to translate"},
                    "l_val": {"type": "string", "description": "Target language code"}
                },
                "required": ["t_val", "l_val"]
            }
        }
    ]
    
    # Test queries
    test_queries = [
        "Calculate 15 * 23 + 5",
        "What's the weather like in Tokyo?",
        "Translate 'Hello world' to French"
    ]
    
    print("\n" + "="*60)
    print("INFERENCE EXAMPLES")
    print("="*60)
    
    for query in test_queries:
        print(f"\n📝 Query: {query}")
        result = orchestrator.select_tool(query, available_tools, temperature=0.1)
        
        print(f"model result: {result}")
        
        if result["success"]:
            plan = result["plan"]
            print(f"🔧 Selected Tool: {plan['tool_use']['tool_name']}")
            print(f"📥 Arguments: {plan['tool_use']['arguments']}")
        else:
            print(f"❌ Failed to parse output")
            print(f"Raw: {result['raw_output'][:500]}...")


if __name__ == "__main__":
    # Run examples
    example_basic_usage()

Loading base model: Qwen/Qwen2.5-1.5B-Instruct


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading LoRA adapter from: /kaggle/working/qwen2.5-orchestrator-10k
Model loaded successfully

INFERENCE EXAMPLES

📝 Query: Calculate 15 * 23 + 5
  [Debug] Raw output preview: {
  "plan_type": "single_step",
  "tool_use": {
    "tool_name": "math_calculator_v1",
    "arguments": {
      "expression": "15 * 23 + 5"
    }
  }
...
  [Debug] Extracted JSON: {
  "plan_type": "single_step",
  "tool_use": {
    "tool_name": "math_calculator_v1",
    "arguments": {
      "expression": "15 * 23 + 5"
    }
  }
}...
  [Debug] Normalizing format: ['plan_type', 'tool_use']
  [Debug] Normalized plan: {'plan_type': 'single_step', 'tool_use': {'tool_name': 'math_calculator_v1', 'arguments': {'expression': '15 * 23 + 5'}}}
model result: {'success': True, 'plan': {'plan_type': 'single_step', 'tool_use': {'tool_name': 'math_calculator_v1', 'arguments': {'expression': '15 * 23 + 5'}}}, 'raw_output': '{\n  "plan_type": "single_step",\n  "tool_use": {\n    "tool_name": "math_calculator_v1",\n    "arguments"

**Inference with real models/functions**

In [4]:
import os
import json
import torch
import re
import math
import random
from datetime import datetime
from typing import List, Dict, Any, Optional, Callable
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# --- ACTUAL TOOL IMPLEMENTATIONS ---

class ToolRegistry:
    """Registry of actual executable tools to test the model's selections."""
    
    def __init__(self):
        self.tools: Dict[str, Callable] = {}
        self.schemas: Dict[str, Dict] = {}
    
    def register(self, name: str, schema: Dict, func: Callable):
        self.tools[name] = func
        self.schemas[name] = schema
        return self
    
    def execute(self, tool_name: str, arguments: Dict) -> Dict[str, Any]:
        """Execute a tool by name with given arguments."""
        if tool_name not in self.tools:
            return {
                "success": False,
                "error": f"Tool '{tool_name}' not found in registry",
                "available": list(self.tools.keys())
            }
        
        tool_func = self.tools[tool_name]
        schema = self.schemas[tool_name]
        
        # Validate required arguments
        required = schema.get("input_schema", {}).get("required", [])
        missing = [arg for arg in required if arg not in arguments]
        if missing:
            return {
                "success": False,
                "error": f"Missing required arguments: {missing}"
            }
        
        try:
            result = tool_func(**arguments)
            return {"success": True, "result": result}
        except Exception as e:
            return {"success": False, "error": str(e)}
    
    def get_available_tools(self) -> List[Dict]:
        """Return tool schemas for prompting."""
        return [
            {
                "name": name,
                "description": schema.get("description", ""),
                "input_schema": schema.get("input_schema", {})
            }
            for name, schema in self.schemas.items()
        ]


# --- ACTUAL TOOL FUNCTIONS ---

def calculator_tool(expression: str) -> Dict[str, Any]:
    """
    SAFE calculator - only evaluates basic math expressions.
    WARNING: In production, use a proper math parser like numexpr or ast.literal_eval with validation.
    """
    # Whitelist allowed characters to prevent code injection
    allowed_pattern = r'^[\d\s\+\-\*\/\(\)\.\,]+$'
    if not re.match(allowed_pattern, expression):
        return {"error": "Invalid characters in expression. Only numbers and +,-,*,/,() allowed."}
    
    try:
        # Basic evaluation (safe for demo - use numexpr in production)
        result = eval(expression, {"__builtins__": {}}, {})
        return {"expression": expression, "result": result, "type": "calculation"}
    except Exception as e:
        return {"error": f"Calculation failed: {str(e)}"}

def weather_tool(city: str) -> Dict[str, Any]:
    """Simulated weather tool (would call real API in production)."""
    # Simulate different weather conditions based on city name hash for consistency
    random.seed(city.lower())
    conditions = ["sunny", "cloudy", "rainy", "snowy", "partly cloudy"]
    temps = {"sunny": (20, 35), "cloudy": (15, 25), "rainy": (10, 20), "snowy": (-5, 5), "partly cloudy": (18, 28)}
    
    condition = random.choice(conditions)
    temp_range = temps[condition]
    temp = random.randint(temp_range[0], temp_range[1])
    
    return {
        "city": city,
        "temperature_c": temp,
        "condition": condition,
        "humidity": random.randint(40, 90),
        "timestamp": datetime.now().isoformat(),
        "note": "Simulated data - integrate OpenWeatherMap API for real data"
    }

def translator_tool(text: str, target_lang: str) -> Dict[str, Any]:
    """Simulated translation tool."""
    translations = {
        "french": {"hello": "bonjour", "world": "monde", "hi": "salut", "goodbye": "au revoir"},
        "spanish": {"hello": "hola", "world": "mundo", "hi": "hola", "goodbye": "adiós"},
        "german": {"hello": "hallo", "world": "welt", "hi": "hallo", "goodbye": "auf wiedersehen"}
    }
    
    lang_key = target_lang.lower().strip()
    text_lower = text.lower().strip(" '\"")
    
    # Simple word-by-word translation for demo
    translated = translations.get(lang_key, {}).get(text_lower, f"[{target_lang} translation of '{text}']")
    
    return {
        "original": text,
        "translated": translated,
        "target_language": target_lang,
        "source_language": "auto-detected",
        "service": "simulated_translator"
    }

def search_tool(query: str) -> Dict[str, Any]:
    """Simulated web search."""
    # Mock search results based on query keywords
    mock_db = {
        "cat": ["Cats are domesticated mammals.", "Cat breeds include Siamese, Persian, Maine Coon."],
        "dog": ["Dogs are domesticated canines.", "Popular breeds: Labrador, German Shepherd."],
        "ai": ["AI stands for Artificial Intelligence.", "Machine learning is a subset of AI."],
        "python": ["Python is a programming language.", "Created by Guido van Rossum in 1991."]
    }
    
    results = []
    query_lower = query.lower()
    for keyword, snippets in mock_db.items():
        if keyword in query_lower:
            results.extend(snippets)
    
    if not results:
        results = [f"No specific results for '{query}'. Try searching for: cats, dogs, AI, python"]
    
    return {
        "query": query,
        "results": results[:3],  # Top 3 results
        "total_found": len(results),
        "search_engine": "mock_search_v1"
    }

def database_tool(query_str: str) -> Dict[str, Any]:
    """Simulated database query executor (read-only)."""
    # Safety check - only allow SELECT statements
    query_upper = query_str.strip().upper()
    if not query_upper.startswith("SELECT"):
        return {"error": "Only SELECT queries allowed for safety", "query": query_str}
    
    # Mock database
    mock_db = {
        "users": [
            {"id": 1, "name": "Alice", "email": "alice@example.com"},
            {"id": 2, "name": "Bob", "email": "bob@example.com"},
            {"id": 3, "name": "Charlie", "email": "charlie@example.com"}
        ],
        "orders": [
            {"id": 99, "user_id": 1, "total": 150.00, "status": "shipped"},
            {"id": 100, "user_id": 2, "total": 230.50, "status": "pending"}
        ]
    }
    
    # Very simple mock SQL parsing
    if "users" in query_str.lower():
        return {"table": "users", "rows": mock_db["users"], "count": len(mock_db["users"])}
    elif "orders" in query_str.lower():
        # Handle WHERE clause simulation
        if "id=99" in query_str or "id = 99" in query_str:
            order = [o for o in mock_db["orders"] if o["id"] == 99]
            return {"table": "orders", "rows": order, "count": len(order)}
        return {"table": "orders", "rows": mock_db["orders"], "count": len(mock_db["orders"])}
    
    return {"error": "Table not found in mock database", "available_tables": list(mock_db.keys())}

def call_donor(donor) -> Dict[str, Any]:
    
    return {
       'donor':donor,
        'call':True
    }
def blood_supply(blood_type) -> Dict[str, Any]:
    
    return {
       'blood_type':blood_type,
        'supply_status':'30%'
    }

# --- ORCHESTRATOR CLASS ---

class RealToolOrchestrator:
    """Orchestrator that actually executes selected tools."""
    
    def __init__(
        self,
        base_model_name: str = "Qwen/Qwen2.5-1.5B-Instruct",
        adapter_path: str = "/kaggle/working/qwen2.5-orchestrator-10k",
        load_in_4bit: bool = True,
        max_new_tokens: int = 512
    ):
        self.max_new_tokens = max_new_tokens
        self.tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        
        print(f"🔧 Loading base model: {base_model_name}")
        
        if load_in_4bit and torch.cuda.is_available():
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
            )
            self.model = AutoModelForCausalLM.from_pretrained(
                base_model_name,
                quantization_config=bnb_config,
                device_map="auto",
                trust_remote_code=True,
                torch_dtype=torch.float16
            )
        else:
            self.model = AutoModelForCausalLM.from_pretrained(
                base_model_name,
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                device_map="auto" if torch.cuda.is_available() else None,
                trust_remote_code=True
            )
        
        if os.path.exists(adapter_path):
            print(f"🔄 Loading LoRA adapter from: {adapter_path}")
            self.model = PeftModel.from_pretrained(self.model, adapter_path)
        else:
            print(f"⚠️ Warning: Adapter not found at {adapter_path}")
        
        self.model.eval()
        print("✅ Model loaded successfully")
        
        # Initialize tool registry with ACTUAL functions
        self.registry = self._setup_tools()
    
    def _setup_tools(self) -> ToolRegistry:
        """Setup registry with executable tools."""
        registry = ToolRegistry()
        
        registry.register(
            name="math_calculator_v1",
            schema={
                "description": "Evaluates mathematical expressions safely.",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "expression": {"type": "string", "description": "Math expression like '15 * 23 + 5'"}
                    },
                    "required": ["expression"]
                }
            },
            func=calculator_tool
        )
        
        registry.register(
            name="weather_get",
            schema={
                "description": "Gets current weather conditions for a city.",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "city": {"type": "string", "description": "City name like 'Tokyo' or 'New York'"}
                    },
                    "required": ["city"]
                }
            },
            func=weather_tool
        )
        
        registry.register(
            name="text_translate",
            schema={
                "description": "Translates text to target language.",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "text": {"type": "string", "description": "Text to translate"},
                        "target_lang": {"type": "string", "description": "Target language like 'French', 'Spanish'"}
                    },
                    "required": ["text", "target_lang"]
                }
            },
            func=translator_tool
        )
        
        registry.register(
            name="web_search",
            schema={
                "description": "Searches the web for information.",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "query": {"type": "string", "description": "Search query string"}
                    },
                    "required": ["query"]
                }
            },
            func=search_tool
        )
        
        registry.register(
            name="db_query",
            schema={
                "description": "Executes SQL SELECT queries against database.",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "query_str": {"type": "string", "description": "SQL query like 'SELECT * FROM users'"}
                    },
                    "required": ["query_str"]
                }
            },
            func=database_tool
        )
        registry.register(
            name="donor-predictor",
            schema={
                "description": "predict whether we should call/text a certain donor",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "donor": {"type": "object", "description": "blood donor information like blood type,age,sex,location "}
                    },
                    "required": ["donor"]
                }
            },
            func=call_donor
        )
        registry.register(
            name="supply-predictor",
            schema={
                "description": "predicts whether the blood supply is good or bad and when its going to run out",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "blood_type": {"type": "string", "description": "blood type 'O-', 'A-','B-','AB-','O+', 'A+','B+','AB+'"}
                    },
                    "required": ["blood_type"]
                }
            },
            func=blood_supply
        )
        
        return registry
    
    def build_prompt(self, user_query: str, tools: List[Dict], system_prompt: Optional[str] = None) -> str:
        default_sys = """You are an intelligent orchestrator.
Analyze the user query and select the appropriate tool from the available tools.
Respond ONLY with a JSON object containing the tool selection and arguments.

TOOLS CONFIGURATION:"""
        
        sys_content = system_prompt if system_prompt else default_sys
        tools_str = json.dumps(tools, indent=2)
        
        messages = [
            {"role": "system", "content": f"{sys_content}\n{tools_str}"},
            {"role": "user", "content": user_query}
        ]
        
        return self.tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=True
        )
    
    def extract_json(self, text: str) -> Optional[str]:
        """Extract JSON from model output."""
        # Try code blocks
        match = re.search(r'```(?:json)?\s*([\s\S]*?)\s*```', text)
        if match:
            return match.group(1)
        
        # Try raw JSON object
        match = re.search(r'(\{[\s\S]*\})', text)
        if match:
            return match.group(1)
        
        return None
    
    def parse_plan(self, text: str) -> Optional[Dict]:
        """Parse model output into structured plan."""
        json_str = self.extract_json(text)
        if not json_str:
            return None
        
        try:
            data = json.loads(json_str)
            
            # Normalize different formats
            if "tool_use" in data:
                return data
            elif "tool_name" in data:
                return {
                    "plan_type": data.get("plan_type", "single_step"),
                    "tool_use": {
                        "tool_name": data["tool_name"],
                        "arguments": data.get("arguments", data.get("kwargs", {}))
                    }
                }
            return None
        except json.JSONDecodeError:
            return None
    
    @torch.no_grad()
    def run(self, user_query: str, temperature: float = 0.1) -> Dict[str, Any]:
        """
        End-to-end execution: parse query -> select tool -> execute tool -> return result.
        """
        # Get available tools
        tools = self.registry.get_available_tools()
        
        # Build prompt
        prompt = self.build_prompt(user_query, tools)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        
        # Generate
        outputs = self.model.generate(
            **inputs,
            max_new_tokens=self.max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )
        
        # Decode
        generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
        generated_text = self.tokenizer.decode(generated_tokens, skip_special_tokens=True)
        
        # Parse plan
        plan = self.parse_plan(generated_text)
        
        if not plan:
            return {
                "success": False,
                "stage": "parsing",
                "error": "Failed to parse model output",
                "raw_output": generated_text,
                "plan": None,
                "execution_result": None
            }
        
        # Extract tool info
        tool_name = plan["tool_use"]["tool_name"]
        arguments = plan["tool_use"]["arguments"]
        
        # Execute the ACTUAL tool
        execution_result = self.registry.execute(tool_name, arguments)
        
        return {
            "success": execution_result["success"] and execution_result.get("success"),
            "stage": "complete",
            "query": user_query,
            "selected_tool": tool_name,
            "arguments": arguments,
            "plan": plan,
            "execution_result": execution_result,
            "raw_output": generated_text
        }


# --- TESTING FRAMEWORK ---

def run_comprehensive_tests():
    """Run tests across all domains with actual execution."""
    
    print("=" * 70)
    print("🧪 COMPREHENSIVE TOOL ORCHESTRATION TESTS")
    print("=" * 70)
    
    orchestrator = RealToolOrchestrator(
        adapter_path="/kaggle/working/qwen2.5-orchestrator-10k"
    )
    
    test_cases = [
        # # Math
        # ("What is 25 divided by 5 plus 10?", "math_calculator_v1"),
        # ("Calculate the square root of 144", "math_calculator_v1"),  # Edge case: sqrt not in whitelist
        
        # # Weather
        # ("What's the temperature in Paris right now?", "weather_get"),
        # ("Will it rain in Seattle today?", "weather_get"),
        
        # # Translation
        # ("Translate 'Good morning' to German", "text_translate"),
        # ("How do you say 'computer' in Spanish?", "text_translate"),
        
        # # Search
        # ("Find information about artificial intelligence", "web_search"),
        # ("Search for cat facts", "web_search"),
        
        # # Database
        # ("Show me all users in the database", "db_query"),
        # ("Get order number 99 details", "db_query"),
        
        # # Ambiguous/Complex
        # ("I need to calculate 100 * 45 and then check the weather in London", "math_calculator_v1"),  # Should pick first or most relevant
        # ("Convert 'hello' to French and tell me about cats", "text_translate"),  # Tests single-step limitation
        
        # Blood-donor-prediction
        ("should call this donor age 90 and has cancer ", "donor-predictor"),  # Should pick first or most relevant
        ("will the O+ blood supply last", "supply-predictor"),  # Tests single-step limitation
    ]
    
    results = {"passed": 0, "failed": 0, "errors": []}
    
    for query, expected_tool in test_cases:
        print(f"\n{'─' * 70}")
        print(f"📝 Query: {query}")
        print(f"🎯 Expected Tool: {expected_tool}")
        
        result = orchestrator.run(query, temperature=0.1)
        
        if not result["success"] and result["stage"] != "complete":
            print(f"❌ FAILED - Stage: {result['stage']}")
            print(f"   Error: {result.get('error', 'Unknown error')}")
            print(f"   Raw: {result['raw_output'][:200]}...")
            results["failed"] += 1
            results["errors"].append({"query": query, "error": result.get("error")})
            continue
        
        selected = result["selected_tool"]
        execution = result["execution_result"]
        
        # Check if correct tool was selected
        tool_correct = (selected == expected_tool)
        
        # Check if execution succeeded
        exec_success = execution.get("success", False)
        
        status = "✅ PASS" if (tool_correct and exec_success) else "❌ FAIL"
        
        print(f"🔧 Selected Tool: {selected} {'✓' if tool_correct else '✗'}")
        print(f"📥 Arguments: {result['arguments']}")
        print(f"⚡ Execution: {'Success' if exec_success else 'Failed'}")
        
        if exec_success:
            print(f"📤 Result: {json.dumps(execution.get('result', {}), indent=2)[:300]}...")
        else:
            print(f"⚠️ Execution Error: {execution.get('error', 'Unknown')}")
        
        print(f"Overall: {status}")
        
        if tool_correct and exec_success:
            results["passed"] += 1
        else:
            results["failed"] += 1
            results["errors"].append({
                "query": query,
                "expected": expected_tool,
                "got": selected,
                "exec_error": execution.get("error") if not exec_success else None
            })
    
    # Summary
    print(f"\n{'=' * 70}")
    print("📊 TEST SUMMARY")
    print(f"{'=' * 70}")
    total = results["passed"] + results["failed"]
    accuracy = (results["passed"] / total * 100) if total > 0 else 0
    
    print(f"Total Tests: {total}")
    print(f"Passed: {results['passed']}")
    print(f"Failed: {results['failed']}")
    print(f"Accuracy: {accuracy:.1f}%")
    
    if results["errors"]:
        print(f"\n⚠️ Errors encountered:")
        for err in results["errors"]:
            print(f"  - {err['query'][:50]}... | {err.get('error') or err.get('got', 'Unknown')}")
    
    return results


def test_edge_cases():
    """Test edge cases and robustness."""
    
    print("\n" + "=" * 70)
    print("🔍 EDGE CASE TESTING")
    print(f"{'=' * 70}")
    
    orchestrator = RealToolOrchestrator(
        adapter_path="/kaggle/working/qwen2.5-orchestrator-10k"
    )
    
    edge_cases = [
        # Obscure tool names (tests generalization)
        {
            "tools": [
                {
                    "name": "svc-9x2a1b",  # UUID style name
                    "description": "Evaluates mathematical expressions.",
                    "input_schema": {
                        "type": "object",
                        "properties": {"expr": {"type": "string"}},
                        "required": ["expr"]
                    }
                },
                {
                    "name": "GlobalWeatherProvider",  # Enterprise style
                    "description": "Returns weather data.",
                    "input_schema": {
                        "type": "object",
                        "properties": {"loc": {"type": "string"}},
                        "required": ["loc"]
                    }
                }
            ],
            "query": "Calculate 5 + 10",
            "expected": "svc-9x2a1b"
        },
        
        # Abstract arguments
        {
            "tools": [
                {
                    "name": "System_X1",
                    "description": "Translates text between languages.",
                    "input_schema": {
                        "type": "object",
                        "properties": {
                            "a": {"type": "string"},  # Abstract arg names
                            "b": {"type": "string"}
                        },
                        "required": ["a", "b"]
                    }
                }
            ],
            "query": "Translate 'hello' to French",
            "expected": "System_X1"
        }
    ]
    
    for i, case in enumerate(edge_cases, 1):
        print(f"\n--- Edge Case {i} ---")
        print(f"Query: {case['query']}")
        print(f"Tools: {[t['name'] for t in case['tools']]}")
        
        # Manually build prompt with custom tools
        prompt = orchestrator.build_prompt(case['query'], case['tools'])
        inputs = orchestrator.tokenizer(prompt, return_tensors="pt").to(orchestrator.model.device)
        
        outputs = orchestrator.model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.1,
            do_sample=False,
            pad_token_id=orchestrator.tokenizer.pad_token_id,
        )
        
        generated = orchestrator.tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:], 
            skip_special_tokens=True
        )
        
        plan = orchestrator.parse_plan(generated)
        
        if plan:
            selected = plan["tool_use"]["tool_name"]
            print(f"Selected: {selected} {'✓' if selected == case['expected'] else '✗'}")
        else:
            print(f"Failed to parse: {generated[:200]}...")


if __name__ == "__main__":
    # Run main tests
    results = run_comprehensive_tests()
    
    # Run edge case tests
    # test_edge_cases()

🧪 COMPREHENSIVE TOOL ORCHESTRATION TESTS
🔧 Loading base model: Qwen/Qwen2.5-1.5B-Instruct


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

🔄 Loading LoRA adapter from: /kaggle/working/qwen2.5-orchestrator-10k
✅ Model loaded successfully

──────────────────────────────────────────────────────────────────────
📝 Query: should call this donor age 90 and has cancer 
🎯 Expected Tool: donor-predictor
🔧 Selected Tool: donor-predictor ✓
📥 Arguments: {'donor': {'blood_type': 'AB-', 'age': 90, 'gender_val': 'M'}}
⚡ Execution: Success
📤 Result: {
  "donor": {
    "blood_type": "AB-",
    "age": 90,
    "gender_val": "M"
  },
  "call": true
}...
Overall: ✅ PASS

──────────────────────────────────────────────────────────────────────
📝 Query: will the O+ blood supply last
🎯 Expected Tool: supply-predictor
🔧 Selected Tool: supply-predictor ✓
📥 Arguments: {'blood_type': 'O+'}
⚡ Execution: Success
📤 Result: {
  "blood_type": "O+",
  "supply_status": "30%"
}...
Overall: ✅ PASS

📊 TEST SUMMARY
Total Tests: 2
Passed: 2
Failed: 0
Accuracy: 100.0%


**Multi-step function calls**

In [2]:
import os
import json
import torch
import re
import math
import random
import asyncio
from datetime import datetime
from typing import List, Dict, Any, Optional, Callable, Union
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# --- ENHANCED TOOL REGISTRY WITH MULTI-STEP SUPPORT ---

class MultiStepToolRegistry:
    """Advanced registry supporting multi-step and parallel execution."""
    
    def __init__(self):
        self.tools: Dict[str, Callable] = {}
        self.schemas: Dict[str, Dict] = {}
        self.execution_history: List[Dict] = []
    
    def register(self, name: str, schema: Dict, func: Callable):
        self.tools[name] = func
        self.schemas[name] = schema
        return self
    
    def execute_single(self, tool_name: str, arguments: Dict, step_id: int = 0) -> Dict[str, Any]:
        """Execute a single tool."""
        if tool_name not in self.tools:
            return {
                "step_id": step_id,
                "tool": tool_name,
                "success": False,
                "error": f"Tool '{tool_name}' not found",
                "result": None
            }
        
        required = self.schemas[tool_name].get("input_schema", {}).get("required", [])
        missing = [arg for arg in required if arg not in arguments]
        if missing:
            return {
                "step_id": step_id,
                "tool": tool_name,
                "success": False,
                "error": f"Missing args: {missing}",
                "result": None
            }
        
        try:
            result = self.tools[tool_name](**arguments)
            execution_record = {
                "step_id": step_id,
                "tool": tool_name,
                "arguments": arguments,
                "success": True,
                "result": result,
                "timestamp": datetime.now().isoformat()
            }
            self.execution_history.append(execution_record)
            return execution_record
        except Exception as e:
            return {
                "step_id": step_id,
                "tool": tool_name,
                "success": False,
                "error": str(e),
                "result": None
            }
    
    def execute_parallel(self, calls: List[Dict]) -> List[Dict]:
        """Execute multiple independent tools in parallel."""
        with ThreadPoolExecutor(max_workers=len(calls)) as executor:
            futures = [
                executor.submit(self.execute_single, call["tool_name"], call["arguments"], i)
                for i, call in enumerate(calls)
            ]
            return [f.result() for f in futures]
    
    def execute_sequential(self, calls: List[Dict]) -> List[Dict]:
        """Execute tools sequentially (for dependent operations)."""
        results = []
        for i, call in enumerate(calls):
            result = self.execute_single(call["tool_name"], call["arguments"], i)
            results.append(result)
            # Make result available for next steps via context
            call["_result"] = result
        return results
    
    def get_available_tools(self) -> List[Dict]:
        return [
            {
                "name": name,
                "description": schema.get("description", ""),
                "input_schema": schema.get("input_schema", {})
            }
            for name, schema in self.schemas.items()
        ]
    
    def clear_history(self):
        self.execution_history = []


# --- ADVANCED TOOL IMPLEMENTATIONS ---

def calculator(expression: str) -> dict:
    """Safe calculator."""
    allowed = r'^[\d\s\+\-\*\/\(\)\.\,]+$'
    if not re.match(allowed, expression):
        return {"error": "Invalid chars", "allowed": "0-9 + - * / ( ) ."}
    try:
        return {"expression": expression, "result": eval(expression, {"__builtins__": {}}, {})}
    except Exception as e:
        return {"error": str(e)}

def weather(city: str) -> dict:
    """Weather tool."""
    random.seed(city.lower())
    conditions = ["sunny", "cloudy", "rainy", "snowy"]
    condition = random.choice(conditions)
    temps = {"sunny": 25, "cloudy": 18, "rainy": 15, "snowy": 2}
    return {
        "city": city,
        "temp_c": temps[condition] + random.randint(-3, 3),
        "condition": condition,
        "humidity": random.randint(40, 90)
    }

def translate(text: str, target_lang: str) -> dict:
    """Translator."""
    simple_map = {
        "french": {"hello": "bonjour", "world": "monde", "good morning": "bonjour"},
        "spanish": {"hello": "hola", "world": "mundo", "good morning": "buenos días"},
        "german": {"hello": "hallo", "world": "welt", "good morning": "guten morgen"}
    }
    lang = target_lang.lower()
    text_lower = text.lower()
    translated = simple_map.get(lang, {}).get(text_lower, f"[{lang}]: {text}")
    return {"original": text, "translated": translated, "lang": target_lang}

def search(query: str) -> dict:
    """Search tool."""
    knowledge = {
        "capital of france": "Paris is the capital of France.",
        "population of tokyo": "Tokyo has approximately 14 million residents.",
        "programming": "Programming is the process of creating computer software.",
        "python": "Python is a high-level programming language."
    }
    query_lower = query.lower()
    for key, value in knowledge.items():
        if key in query_lower:
            return {"query": query, "answer": value, "source": "knowledge_base"}
    return {"query": query, "answer": "No direct match found.", "source": "none"}

def currency_convert(amount: float, from_currency: str, to_currency: str) -> dict:
    """Currency converter."""
    rates = {"USD": 1.0, "EUR": 0.92, "GBP": 0.79, "JPY": 150.0}
    if from_currency not in rates or to_currency not in rates:
        return {"error": "Unsupported currency", "supported": list(rates.keys())}
    converted = amount * (rates[to_currency] / rates[from_currency])
    return {
        "amount": amount,
        "from": from_currency,
        "to": to_currency,
        "result": round(converted, 2),
        "rate": round(rates[to_currency] / rates[from_currency], 4)
    }

def format_text(text: str, format_type: str) -> dict:
    """Text formatter."""
    if format_type == "uppercase":
        return {"text": text.upper(), "format": format_type}
    elif format_type == "lowercase":
        return {"text": text.lower(), "format": format_type}
    elif format_type == "reverse":
        return {"text": text[::-1], "format": format_type}
    return {"error": "Unknown format", "supported": ["uppercase", "lowercase", "reverse"]}


# --- MULTI-STEP ORCHESTRATOR ---

class MultiStepOrchestrator:
    """Orchestrator supporting multi-step planning and execution."""
    
    PLAN_TYPES = {
        "single_step": "One tool call",
        "sequential": "Multiple dependent steps",
        "parallel": "Multiple independent steps simultaneously",
        "conditional": "Branch based on previous result"
    }
    
    def __init__(
        self,
        base_model_name: str = "Qwen/Qwen2.5-1.5B-Instruct",
        adapter_path: str = "/kaggle/working/qwen2.5-orchestrator-10k",
        load_in_4bit: bool = True
    ):
        self.tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        
        print(f"🔧 Loading model: {base_model_name}")
        
        if load_in_4bit and torch.cuda.is_available():
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
            )
            self.model = AutoModelForCausalLM.from_pretrained(
                base_model_name,
                quantization_config=bnb_config,
                device_map="auto",
                trust_remote_code=True
            )
        else:
            self.model = AutoModelForCausalLM.from_pretrained(
                base_model_name,
                device_map="auto" if torch.cuda.is_available() else None,
                trust_remote_code=True
            )
        
        if os.path.exists(adapter_path):
            print(f"🔄 Loading adapter: {adapter_path}")
            self.model = PeftModel.from_pretrained(self.model, adapter_path)
        
        self.model.eval()
        
        self.registry = self._setup_registry()
        print("✅ Ready for multi-step orchestration")
    
    def _setup_registry(self) -> MultiStepToolRegistry:
        reg = MultiStepToolRegistry()
        reg.register("calculator", {
            "description": "Calculates mathematical expressions",
            "input_schema": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]}
        }, calculator)
        
        reg.register("weather", {
            "description": "Gets weather for a city",
            "input_schema": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}
        }, weather)
        
        reg.register("translate", {
            "description": "Translates text to target language",
            "input_schema": {"type": "object", "properties": {"text": {"type": "string"}, "target_lang": {"type": "string"}}, "required": ["text", "target_lang"]}
        }, translate)
        
        reg.register("search", {
            "description": "Searches for information",
            "input_schema": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}
        }, search)
        
        reg.register("currency_convert", {
            "description": "Converts currency amounts",
            "input_schema": {"type": "object", "properties": {"amount": {"type": "number"}, "from_currency": {"type": "string"}, "to_currency": {"type": "string"}}, "required": ["amount", "from_currency", "to_currency"]}
        }, currency_convert)
        
        reg.register("format_text", {
            "description": "Formats text (uppercase, lowercase, reverse)",
            "input_schema": {"type": "object", "properties": {"text": {"type": "string"}, "format_type": {"type": "string"}}, "required": ["text", "format_type"]}
        }, format_text)
        
        return reg
    
    def build_prompt(self, query: str, tools: List[Dict], allow_multi_step: bool = True) -> str:
        multi_step_instruction = ""
        if allow_multi_step:
            multi_step_instruction = """
You can respond with MULTIPLE tool calls if needed:
- Use "plan_type": "parallel" for independent operations that can run simultaneously
- Use "plan_type": "sequential" for dependent operations that must run in order
- Use "plan_type": "single_step" for just one tool"""
        
        system_msg = f"""You are an intelligent orchestrator. Analyze the user request and select appropriate tool(s).
Respond with JSON containing the execution plan.{multi_step_instruction}

Available Tools:"""
        
        messages = [
            {"role": "system", "content": f"{system_msg}\n{json.dumps(tools, indent=2)}"},
            {"role": "user", "content": query}
        ]
        
        return self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    def parse_multi_step_output(self, text: str) -> Optional[Dict]:
        """Parse single or multi-step plans."""
        # Extract JSON
        json_match = re.search(r'```(?:json)?\s*([\s\S]*?)\s*```', text)
        json_str = json_match.group(1) if json_match else re.search(r'(\{[\s\S]*\})', text)
        if not json_str:
            return None
        
        try:
            if isinstance(json_str, re.Match):
                json_str = json_str.group(1)
            data = json.loads(json_str)
            
            # Normalize various formats
            if "plan_type" not in data:
                # Single tool flat format
                if "tool_name" in data:
                    return {
                        "plan_type": "single_step",
                        "steps": [{
                            "tool_name": data["tool_name"],
                            "arguments": data.get("arguments", data.get("kwargs", {}))
                        }]
                    }
                return None
            
            # Handle multi-step formats
            plan_type = data["plan_type"]
            
            if plan_type == "single_step":
                tool_use = data.get("tool_use", {})
                return {
                    "plan_type": "single_step",
                    "steps": [{
                        "tool_name": tool_use.get("tool_name", data.get("tool_name")),
                        "arguments": tool_use.get("arguments", data.get("arguments", {}))
                    }]
                }
            
            elif plan_type in ["sequential", "parallel"]:
                steps = data.get("steps", [])
                if not steps and "tool_calls" in data:
                    steps = data["tool_calls"]
                return {
                    "plan_type": plan_type,
                    "steps": steps
                }
            
            return data
            
        except json.JSONDecodeError as e:
            print(f"Parse error: {e}")
            return None
    
    @torch.no_grad()
    def orchestrate(self, query: str, allow_multi_step: bool = True, temperature: float = 0.1) -> Dict[str, Any]:
        """Full orchestration pipeline."""
        tools = self.registry.get_available_tools()
        prompt = self.build_prompt(query, tools, allow_multi_step)
        
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        
        outputs = self.model.generate(
            **inputs,
            max_new_tokens=1024,  # Larger for multi-step
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )
        
        generated = self.tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        plan = self.parse_multi_step_output(generated)
        
        if not plan:
            return {
                "success": False,
                "error": "Failed to parse plan",
                "raw_output": generated,
                "execution_results": None
            }
        
        # Execute based on plan type
        steps = plan.get("steps", [])
        plan_type = plan.get("plan_type", "single_step")
        
        if plan_type == "parallel" and len(steps) > 1:
            execution_results = self.registry.execute_parallel(steps)
        else:
            execution_results = self.registry.execute_sequential(steps)
        
        success = all(r["success"] for r in execution_results)
        
        return {
            "success": success,
            "query": query,
            "plan_type": plan_type,
            "steps_planned": len(steps),
            "steps": steps,
            "execution_results": execution_results,
            "raw_output": generated,
            "all_succeeded": success
        }


# --- TEST SUITE FOR MULTI-STEP ---

def test_multi_step_orchestration():
    """Comprehensive multi-step testing."""
    
    print("=" * 80)
    print("🚀 MULTI-STEP ORCHESTRATION TEST SUITE")
    print("=" * 80)
    
    orchestrator = MultiStepOrchestrator()
    
    test_cases = [
        # Single step (baseline)
        {
            "name": "Single Math",
            "query": "Calculate 15 * 8",
            "expect_parallel": False,
            "min_steps": 1,
            "max_steps": 1
        },
        
        # Implicit multi-step (should detect need for multiple calls)
        {
            "name": "Weather Comparison",
            "query": "Compare the weather in Paris and Tokyo",
            "expect_parallel": True,  # Independent calls
            "min_steps": 2,
            "max_steps": 3
        },
        
        # Parallel execution
        {
            "name": "Multi-Currency Convert",
            "query": "Convert 100 USD to EUR and also convert 50 GBP to JPY",
            "expect_parallel": True,
            "min_steps": 2,
            "max_steps": 2
        },
        
        # Sequential dependency (if model supports it)
        {
            "name": "Calculate and Format",
            "query": "Calculate 5 * 10 and then convert the result to uppercase text",
            "expect_parallel": False,  # Dependent steps
            "min_steps": 2,
            "max_steps": 2
        },
        
        # Complex query requiring breakdown
        {
            "name": "Travel Planner",
            "query": "I need to know the weather in London, convert 500 USD to GBP, and translate 'hello' to English",
            "expect_parallel": True,  # All independent
            "min_steps": 2,
            "max_steps": 4
        },
        
        # Search + Calculate
        {
            "name": "Research and Calculate",
            "query": "Search for the population of Tokyo and calculate how many people per square kilometer if the area is 2194 km²",
            "expect_parallel": False,  # Dependent (search first, then calc)
            "min_steps": 2,
            "max_steps": 2
        },
        
        # Stress test: Many operations
        {
            "name": "Mass Parallel",
            "query": "Get weather for New York, London, Paris, Tokyo, and Sydney all at once",
            "expect_parallel": True,
            "min_steps": 4,
            "max_steps": 6
        }
    ]
    
    results = []
    
    for test in test_cases:
        print(f"\n{'─' * 80}")
        print(f"📋 TEST: {test['name']}")
        print(f"📝 Query: {test['query']}")
        
        result = orchestrator.orchestrate(test['query'], allow_multi_step=True)
        
        # Validate
        planned_steps = len(result.get("steps", []))
        is_parallel = result.get("plan_type") == "parallel"
        all_success = result.get("all_succeeded", False)
        
        # Check expectations
        step_ok = test['min_steps'] <= planned_steps <= test['max_steps']
        parallel_ok = (is_parallel == test['expect_parallel']) or not test['expect_parallel']
        
        passed = result['success'] and step_ok and all_success
        
        status = "✅ PASS" if passed else "❌ FAIL"
        
        print(f"🔧 Plan Type: {result.get('plan_type', 'unknown')}")
        print(f"📊 Steps: {planned_steps} (expected {test['min_steps']}-{test['max_steps']})")
        print(f"⚡ Execution: {'All succeeded' if all_success else 'Some failed'}")
        print(f"🔄 Parallel: {is_parallel} (expected: {test['expect_parallel']})")
        
        if result['execution_results']:
            for i, exec_result in enumerate(result['execution_results'][:3]):  # Show first 3
                tool = exec_result['tool']
                success = "✓" if exec_result['success'] else "✗"
                res_summary = str(exec_result.get('result', {}))[:60]
                print(f"   Step {i+1}: {tool} {success} -> {res_summary}...")
        
        print(f"Overall: {status}")
        
        results.append({
            "name": test['name'],
            "passed": passed,
            "plan_type": result.get('plan_type'),
            "steps": planned_steps,
            "expected_parallel": test['expect_parallel'],
            "was_parallel": is_parallel
        })
        
        # Clear history between tests
        orchestrator.registry.clear_history()
    
    # Summary
    print(f"\n{'=' * 80}")
    print("📊 MULTI-STEP TEST SUMMARY")
    print(f"{'=' * 80}")
    
    passed = sum(1 for r in results if r['passed'])
    total = len(results)
    
    for r in results:
        icon = "✅" if r['passed'] else "❌"
        print(f"{icon} {r['name']}: {r['plan_type']} ({r['steps']} steps)")
    
    print(f"\nTotal: {passed}/{total} ({passed/total*100:.0f}%)")
    
    # Analysis
    parallel_tests = [r for r in results if r['expected_parallel']]
    if parallel_tests:
        parallel_correct = sum(1 for r in parallel_tests if r['was_parallel'] == r['expected_parallel'])
        print(f"Parallel Detection Accuracy: {parallel_correct}/{len(parallel_tests)}")


def test_single_vs_multi_comparison():
    """Compare single-step vs multi-step prompting on same queries."""
    
    print("\n" + "=" * 80)
    print("🔄 SINGLE vs MULTI-STEP COMPARISON")
    print(f"{'=' * 80}")
    
    orchestrator = MultiStepOrchestrator()
    
    queries = [
        "Convert 100 USD to EUR and check weather in Paris",
        "Calculate 10 + 5 and translate 'hello' to Spanish"
    ]
    
    for query in queries:
        print(f"\n📝 Query: {query}")
        
        # Single-step mode
        result_single = orchestrator.orchestrate(query, allow_multi_step=False)
        steps_single = len(result_single.get('steps', []))
        
        # Multi-step mode
        result_multi = orchestrator.orchestrate(query, allow_multi_step=True)
        steps_multi = len(result_multi.get('steps', []))
        
        print(f"   Single-step mode: {steps_single} step(s)")
        print(f"   Multi-step mode:  {steps_multi} step(s) ({result_multi.get('plan_type')})")
        
        if steps_multi > steps_single:
            print("   📈 Multi-step planning utilized!")
        orchestrator.registry.clear_history()


if __name__ == "__main__":
    test_multi_step_orchestration()
    test_single_vs_multi_comparison()

🚀 MULTI-STEP ORCHESTRATION TEST SUITE


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

🔄 Loading adapter: /kaggle/working/qwen2.5-orchestrator-10k
✅ Ready for multi-step orchestration

────────────────────────────────────────────────────────────────────────────────
📋 TEST: Single Math
📝 Query: Calculate 15 * 8
🔧 Plan Type: single_step
📊 Steps: 1 (expected 1-1)
⚡ Execution: Some failed
🔄 Parallel: False (expected: False)
   Step 1: calc_main_lib ✗ -> None...
Overall: ❌ FAIL

────────────────────────────────────────────────────────────────────────────────
📋 TEST: Weather Comparison
📝 Query: Compare the weather in Paris and Tokyo
🔧 Plan Type: single_step
📊 Steps: 1 (expected 2-3)
⚡ Execution: All succeeded
🔄 Parallel: False (expected: True)
   Step 1: weather ✓ -> {'city': 'Paris', 'temp_c': 2, 'condition': 'snowy', 'humidi...
Overall: ❌ FAIL

────────────────────────────────────────────────────────────────────────────────
📋 TEST: Multi-Currency Convert
📝 Query: Convert 100 USD to EUR and also convert 50 GBP to JPY
🔧 Plan Type: single_step
📊 Steps: 1 (expected 2-2)
⚡ Execut

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

🔄 Loading adapter: /kaggle/working/qwen2.5-orchestrator-10k
✅ Ready for multi-step orchestration

📝 Query: Convert 100 USD to EUR and check weather in Paris
   Single-step mode: 1 step(s)
   Multi-step mode:  1 step(s) (single_step)

📝 Query: Calculate 10 + 5 and translate 'hello' to Spanish
   Single-step mode: 1 step(s)
   Multi-step mode:  1 step(s) (single_step)


**Multi-step dataset geneartor**

In [ ]:
import json
import random
import uuid
from typing import List, Dict, Any, Tuple
from tqdm import tqdm

# --- MULTI-STEP DATASET GENERATOR ---

class MultiStepDatasetGenerator:
    """
    Generates training data for multi-step tool orchestration.
    Creates examples with parallel, sequential, and conditional plans.
    """
    
    def __init__(self, seed: int = 42):
        random.seed(seed)
        
        # Tool definitions with varied naming (to prevent overfitting)
        self.tool_templates = {
            "calculator": {
                "concepts": ["Math", "Calc", "Compute", "Number"],
                "actions": ["calculate", "compute", "solve", "evaluate"],
                "args": [("expression", "string", "Math expression to evaluate")],
                "examples": [("15 * 8", "120"), ("sqrt(144)", "12"), ("5 + 10", "15")]
            },
            "weather": {
                "concepts": ["Weather", "Meteo", "Climate", "Sky"],
                "actions": ["get weather", "check forecast", "temperature in"],
                "args": [("city", "string", "City name")],
                "examples": [("Paris", "15°C sunny"), ("Tokyo", "22°C cloudy")]
            },
            "translate": {
                "concepts": ["Translate", "Lang", "Linguist"],
                "actions": ["translate", "convert to", "say in"],
                "args": [
                    ("text", "string", "Text to translate"),
                    ("target_lang", "string", "Target language")
                ],
                "examples": [("hello", "French", "bonjour"), ("world", "Spanish", "mundo")]
            },
            "currency": {
                "concepts": ["Currency", "Money", "Finance", "Exchange"],
                "actions": ["convert", "exchange", "rate for"],
                "args": [
                    ("amount", "number", "Amount to convert"),
                    ("from_currency", "string", "Source currency code"),
                    ("to_currency", "string", "Target currency code")
                ],
                "examples": [(100, "USD", "EUR", 92.0), (50, "GBP", "JPY", 7500)]
            },
            "search": {
                "concepts": ["Search", "Find", "Lookup", "Query"],
                "actions": ["search for", "find information about", "look up"],
                "args": [("query", "string", "Search query")],
                "examples": [("capital of France", "Paris"), ("Python programming", "Python is a language")]
            },
            "format": {
                "concepts": ["Format", "Text", "String", "Transform"],
                "actions": ["format", "convert to", "make"],
                "args": [
                    ("text", "string", "Input text"),
                    ("format_type", "string", "Format: uppercase, lowercase, reverse")
                ],
                "examples": [("hello", "uppercase", "HELLO"), ("WORLD", "lowercase", "world")]
            }
        }
        
    def generate_tool_name(self, concept: str) -> str:
        """Generate varied tool names."""
        strategies = [
            lambda c: f"{c.lower()}_v{random.randint(1,3)}",
            lambda c: f"svc-{uuid.uuid4().hex[:6]}",
            lambda c: f"Global{c}Provider",
            lambda c: f"System_{random.choice(['A','B','X','Z'])}{random.randint(1,99)}",
            lambda c: f"{c}Service",
            lambda c: f"{c.lower()}_api"
        ]
        return random.choice(strategies)(concept)
    
    def generate_arg_name(self, base_name: str) -> str:
        """Generate varied argument names."""
        variations = {
            "expression": ["expr", "formula", "eq", "math_str", "calculation"],
            "city": ["loc", "place", "c", "destination", "target_city"],
            "text": ["t", "content", "input_str", "source_text", "txt"],
            "target_lang": ["lang", "target", "l", "to_language", "dest_lang"],
            "amount": ["amt", "value", "sum", "a", "money"],
            "from_currency": ["from", "src_curr", "fc", "source"],
            "to_currency": ["to", "dst_curr", "tc", "dest"],
            "query": ["q", "search_term", "keyword", "question"],
            "format_type": ["fmt", "type", "style", "format_style"]
        }
        
        if base_name in variations:
            return random.choice([base_name] + variations[base_name])
        return base_name
    
    def create_tool_schema(self, tool_type: str) -> Tuple[str, Dict]:
        """Create a tool schema with randomized names."""
        template = self.tool_templates[tool_type]
        base_concept = random.choice(template["concepts"])
        name = self.generate_tool_name(base_concept)
        
        properties = {}
        arg_mapping = {}  # Maps canonical arg names to generated names
        
        for canonical_arg, arg_type, description in template["args"]:
            generated_arg = self.generate_arg_name(canonical_arg)
            arg_mapping[canonical_arg] = generated_arg
            properties[generated_arg] = {
                "type": arg_type,
                "description": description
            }
        
        schema = {
            "name": name,
            "description": random.choice([
                f"Tool for {tool_type} operations.",
                f"Handles {tool_type} requests.",
                f"Specialized {tool_type} service.",
                f"Performs {tool_type} functionality."
            ]),
            "input_schema": {
                "type": "object",
                "properties": properties,
                "required": list(properties.keys())
            }
        }
        
        return name, schema, arg_mapping, template
    
    def generate_single_step(self) -> Dict:
        """Generate a single-step example (baseline)."""
        tool_type = random.choice(list(self.tool_templates.keys()))
        name, schema, arg_map, template = self.create_tool_schema(tool_type)
        
        # Generate query and arguments
        example_data = random.choice(template["examples"])
        query_parts = []
        arguments = {}
        
        if tool_type == "calculator":
            expr = example_data[0]
            query = f"Calculate {expr}"
            arguments[arg_map["expression"]] = expr
            
        elif tool_type == "weather":
            city = example_data[0]
            query = f"What's the weather in {city}?"
            arguments[arg_map["city"]] = city
            
        elif tool_type == "translate":
            text, lang, _ = example_data
            query = f"Translate '{text}' to {lang}"
            arguments[arg_map["text"]] = text
            arguments[arg_map["target_lang"]] = lang
            
        elif tool_type == "currency":
            amt, frm, to, _ = example_data
            query = f"Convert {amt} {frm} to {to}"
            arguments[arg_map["amount"]] = amt
            arguments[arg_map["from_currency"]] = frm
            arguments[arg_map["to_currency"]] = to
            
        elif tool_type == "search":
            q = example_data[0]
            query = f"Search for {q}"
            arguments[arg_map["query"]] = q
            
        elif tool_type == "format":
            text, fmt, _ = example_data
            query = f"Format '{text}' as {fmt}"
            arguments[arg_map["text"]] = text
            arguments[arg_map["format_type"]] = fmt
        
        # Add distractors
        all_tools = [schema] + self._generate_distractors([tool_type])
        random.shuffle(all_tools)
        
        return self._build_message(
            query=query,
            tools=all_tools,
            plan_type="single_step",
            steps=[{"tool_name": name, "arguments": arguments}]
        )
    
    def generate_parallel(self) -> Dict:
        """
        Generate parallel execution example.
        Multiple independent tool calls that can run simultaneously.
        """
        # Select 2-3 unrelated tools
        num_tools = random.randint(2, 3)
        selected_types = random.sample(list(self.tool_templates.keys()), num_tools)
        
        steps = []
        query_parts = []
        all_tool_schemas = []
        
        for tool_type in selected_types:
            name, schema, arg_map, template = self.create_tool_schema(tool_type)
            all_tool_schemas.append(schema)
            
            example_data = random.choice(template["examples"])
            
            if tool_type == "weather":
                city = example_data[0]
                query_parts.append(f"weather in {city}")
                steps.append({
                    "tool_name": name,
                    "arguments": {arg_map["city"]: city}
                })
                
            elif tool_type == "currency":
                amt, frm, to, _ = example_data
                query_parts.append(f"convert {amt} {frm} to {to}")
                steps.append({
                    "tool_name": name,
                    "arguments": {
                        arg_map["amount"]: amt,
                        arg_map["from_currency"]: frm,
                        arg_map["to_currency"]: to
                    }
                })
                
            elif tool_type == "translate":
                text, lang, _ = example_data
                query_parts.append(f"translate '{text}' to {lang}")
                steps.append({
                    "tool_name": name,
                    "arguments": {
                        arg_map["text"]: text,
                        arg_map["target_lang"]: lang
                    }
                })
                
            elif tool_type == "calculator":
                expr = example_data[0]
                query_parts.append(f"calculate {expr}")
                steps.append({
                    "tool_name": name,
                    "arguments": {arg_map["expression"]: expr}
                })
                
            elif tool_type == "search":
                q = example_data[0]
                query_parts.append(f"search for {q}")
                steps.append({
                    "tool_name": name,
                    "arguments": {arg_map["query"]: q}
                })
                
            elif tool_type == "format":
                text, fmt, _ = example_data
                query_parts.append(f"format '{text}' to {fmt}")
                steps.append({
                    "tool_name": name,
                    "arguments": {
                        arg_map["text"]: text,
                        arg_map["format_type"]: fmt
                    }
                })
        
        # Build query with "and", "also", "plus"
        connectors = [" and ", ", ", " plus ", " also "]
        random.shuffle(query_parts)
        query = "I need to " + connectors[0].join(query_parts)
        
        # Add distractors
        all_tools = all_tool_schemas + self._generate_distractors(selected_types)
        random.shuffle(all_tools)
        
        return self._build_message(
            query=query,
            tools=all_tools,
            plan_type="parallel",
            steps=steps
        )
    
    def generate_sequential(self) -> Dict:
        """
        Generate sequential execution example.
        Steps that depend on previous results.
        """
        # Common pattern: Search -> Calculate or Calculate -> Format
        pattern = random.choice(["search_calc", "calc_format"])
        
        if pattern == "search_calc":
            # Search for data then calculate with it
            search_name, search_schema, search_arg_map, _ = self.create_tool_schema("search")
            calc_name, calc_schema, calc_arg_map, _ = self.create_tool_schema("calculator")
            
            query = "Search for population of Tokyo and calculate density for area 2194 km²"
            
            steps = [
                {
                    "tool_name": search_name,
                    "arguments": {search_arg_map["query"]: "population of Tokyo"},
                    "store_result_as": "population_data"
                },
                {
                    "tool_name": calc_name,
                    "arguments": {
                        calc_arg_map["expression"]: "13960000 / 2194"  # Tokyo population / area
                    },
                    "depends_on": "population_data"
                }
            ]
            
            all_tools = [search_schema, calc_schema] + self._generate_distractors(["search", "calculator"])
            
        else:  # calc_format
            # Calculate then format result
            calc_name, calc_schema, calc_arg_map, _ = self.create_tool_schema("calculator")
            fmt_name, fmt_schema, fmt_arg_map, _ = self.create_tool_schema("format")
            
            expr = "15 * 8"
            query = f"Calculate {expr} and format the result as uppercase"
            
            steps = [
                {
                    "tool_name": calc_name,
                    "arguments": {calc_arg_map["expression"]: expr},
                    "store_result_as": "calc_result"
                },
                {
                    "tool_name": fmt_name,
                    "arguments": {
                        fmt_arg_map["text"]: "120",  # Result of 15*8
                        fmt_arg_map["format_type"]: "uppercase"
                    },
                    "depends_on": "calc_result"
                }
            ]
            
            all_tools = [calc_schema, fmt_schema] + self._generate_distractors(["calculator", "format"])
        
        random.shuffle(all_tools)
        
        return self._build_message(
            query=query,
            tools=all_tools,
            plan_type="sequential",
            steps=steps
        )
    
    def generate_conditional(self) -> Dict:
        """
        Generate conditional execution example.
        Branch based on conditions (advanced).
        """
        weather_name, weather_schema, weather_arg_map, _ = self.create_tool_schema("weather")
        calc_name, calc_schema, calc_arg_map, _ = self.create_tool_schema("calculator")
        
        city = random.choice(["Paris", "London", "New York"])
        query = f"Check weather in {city}. If temperature is below 10°C, calculate 10 + 5"
        
        steps = [
            {
                "tool_name": weather_name,
                "arguments": {weather_arg_map["city"]: city},
                "condition": {
                    "if": "temp < 10",
                    "then": {
                        "tool_name": calc_name,
                        "arguments": {calc_arg_map["expression"]: "10 + 5"}
                    }
                }
            }
        ]
        
        all_tools = [weather_schema, calc_schema] + self._generate_distractors(["weather", "calculator"])
        random.shuffle(all_tools)
        
        return self._build_message(
            query=query,
            tools=all_tools,
            plan_type="conditional",
            steps=steps
        )
    
    def _generate_distractors(self, exclude_types: List[str], num: int = 3) -> List[Dict]:
        """Generate distractor tools."""
        distractors = []
        available = [t for t in self.tool_templates.keys() if t not in exclude_types]
        
        for _ in range(min(num, len(available))):
            tool_type = random.choice(available)
            _, schema, _, _ = self.create_tool_schema(tool_type)
            distractors.append(schema)
        
        return distractors
    
    def _build_message(self, query: str, tools: List[Dict], plan_type: str, steps: List[Dict]) -> Dict:
        """Build the final training message."""
        system_msg = f"""You are an intelligent orchestrator.
Analyze the user request and determine if it requires single or multiple tool calls.
Respond with a plan indicating execution strategy.

Plan Types:
- single_step: One independent tool call
- parallel: Multiple independent calls that can run simultaneously
- sequential: Multiple dependent calls that must run in order
- conditional: Execution depends on previous results

TOOLS CONFIGURATION:
{json.dumps(tools, indent=2)}"""
        
        plan_output = {
            "plan_type": plan_type,
            "steps": steps
        }
        
        return {
            "messages": [
                {"role": "system", "content": system_msg},
                {"role": "user", "content": query},
                {"role": "assistant", "content": json.dumps(plan_output, indent=2)}
            ]
        }
    
    def generate_dataset(self, total_size: int = 10000, output_file: str = "multistep_dataset.json") -> None:
        """Generate full dataset with mixed plan types."""
        dataset = []
        
        # Distribution: 40% single, 30% parallel, 20% sequential, 10% conditional
        distribution = {
            "single": int(total_size * 0.4),
            "parallel": int(total_size * 0.3),
            "sequential": int(total_size * 0.2),
            "conditional": int(total_size * 0.1)
        }
        
        generators = {
            "single": self.generate_single_step,
            "parallel": self.generate_parallel,
            "sequential": self.generate_sequential,
            "conditional": self.generate_conditional
        }
        
        for plan_type, count in distribution.items():
            print(f"Generating {count} {plan_type} examples...")
            for _ in tqdm(range(count), desc=plan_type):
                dataset.append(generators[plan_type]())
        
        # Shuffle dataset
        random.shuffle(dataset)
        
        # Save
        with open(output_file, 'w') as f:
            json.dump(dataset, f, indent=2)
        
        print(f"\n✅ Generated {len(dataset)} examples -> {output_file}")
        print(f"Distribution: {distribution}")


# --- USAGE ---

if __name__ == "__main__":
    generator = MultiStepDatasetGenerator(seed=42)
    generator.generate_dataset(total_size=10000, output_file="multistep_10k.json")
    
    # Show sample examples
    print("\n" + "="*60)
    print("SAMPLE EXAMPLES:")
    print("="*60)
    
    examples = [
        generator.generate_single_step(),
        generator.generate_parallel(),
        generator.generate_sequential(),
        generator.generate_conditional()
    ]
    
    for i, ex in enumerate(examples, 1):
        print(f"\n--- Example {i} ---")
        print(json.dumps(ex, indent=2))

**Loading and Using saved models**

In [7]:
import os
import json
import pickle
import torch
import re
from pathlib import Path
from typing import List, Dict, Any, Optional, Callable, Union
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import pandas as pd
import numpy as np

# --- DYNAMIC MODEL LOADER ---

class ModelRegistry:
    """Registry that loads and manages ML models from .pkl files based on config."""
    
    def __init__(self, config_path: str):
        self.config_path = Path(config_path)
        self.models_dir = self.config_path.parent
        self.config = self._load_config()
        self.loaded_models: Dict[str, Any] = {}
        self.model_schemas: Dict[str, Dict] = {}
        
    def _load_config(self) -> Dict:
        """Load model configuration file."""
        with open(self.config_path, 'r') as f:
            return json.load(f)
    
    def load_all_models(self) -> Dict[str, Callable]:
        """Load all models defined in config and return as tool functions."""
        tools = {}
        
        for model_config in self.config.get("models", []):
            model_id = model_config["id"]
            model_type = model_config.get("type", "sklearn")
            model_file = model_config["file_path"]
            model_path = self.models_dir / model_file
            
            print(f"🔧 Loading model: {model_id} from {model_path}")
            
            if not model_path.exists():
                print(f"⚠️ Model file not found: {model_path}")
                continue
            
            try:
                # Load the pickle file
                with open(model_path, 'rb') as f:
                    model_obj = pickle.load(f)
                
                self.loaded_models[model_id] = {
                    "model": model_obj,
                    "config": model_config,
                    "type": model_type
                }
                
                # Create tool wrapper function
                tool_func = self._create_tool_wrapper(model_id, model_config)
                tools[model_id] = tool_func
                
                # Store schema for orchestrator
                self.model_schemas[model_id] = self._build_schema(model_config)
                
                print(f"✅ Loaded {model_id} ({model_type})")
                
            except Exception as e:
                print(f"❌ Failed to load {model_id}: {e}")
        
        return tools
    
    def _create_tool_wrapper(self, model_id: str, config: Dict) -> Callable:
        """Create a callable tool function for the model."""
        model_entry = self.loaded_models[model_id]
        model_obj = model_entry["model"]
        model_type = model_entry["type"]
        
        def tool_function(**kwargs) -> Dict[str, Any]:
            """Dynamic tool function that adapts to model type."""
            try:
                if model_type == "sklearn":
                    return self._predict_sklearn(model_obj, config, kwargs)
                elif model_type == "pytorch":
                    return self._predict_pytorch(model_obj, config, kwargs)
                elif model_type == "custom":
                    return self._predict_custom(model_obj, config, kwargs)
                else:
                    return {"error": f"Unknown model type: {model_type}"}
            except Exception as e:
                return {"error": str(e), "model": model_id}
        
        # Preserve metadata
        tool_function.__name__ = model_id
        tool_function.__doc__ = config.get("description", f"ML model: {model_id}")
        return tool_function
    
    def _predict_sklearn(self, model, config: Dict, inputs: Dict) -> Dict:
        """Handle sklearn model predictions."""
        # Map input fields to features
        features = config.get("features", [])
        input_order = config.get("input_order", features)
        
        # Build feature vector
        feature_vector = []
        for feat in input_order:
            if feat in inputs:
                feature_vector.append(inputs[feat])
            else:
                # Handle missing with default or error
                default = config.get("defaults", {}).get(feat, 0)
                feature_vector.append(default)
        
        X = np.array(feature_vector).reshape(1, -1)
        
        # Predict
        prediction = model.predict(X)[0]
        
        # Get probabilities if available
        probabilities = None
        if hasattr(model, "predict_proba"):
            probabilities = model.predict_proba(X)[0].tolist()
        
        return {
            "prediction": prediction,
            "probabilities": probabilities,
            "input_features": dict(zip(input_order, feature_vector)),
            "model_type": "sklearn"
        }
    
    def _predict_pytorch(self, model, config: Dict, inputs: Dict) -> Dict:
        """Handle PyTorch model predictions."""
        features = config.get("features", [])
        input_order = config.get("input_order", features)
        
        # Build tensor
        feature_vector = [inputs.get(f, config.get("defaults", {}).get(f, 0)) for f in input_order]
        X = torch.tensor([feature_vector], dtype=torch.float32)
        
        model.eval()
        with torch.no_grad():
            output = model(X)
            
        # Handle different output types
        if output.dim() > 1:
            prediction = output.argmax(dim=1).item() if output.shape[1] > 1 else output.item()
            probabilities = torch.softmax(output, dim=1).tolist() if output.shape[1] > 1 else None
        else:
            prediction = output.item()
            probabilities = None
        
        return {
            "prediction": prediction,
            "probabilities": probabilities,
            "raw_output": output.tolist(),
            "model_type": "pytorch"
        }
    
    def _predict_custom(self, model, config: Dict, inputs: Dict) -> Dict:
        """Handle custom model objects (like your DonorPredictionService)."""
        # Check if model has predict method
        if hasattr(model, "predict"):
            prediction = model.predict(inputs)
            return {"prediction": prediction, "model_type": "custom"}
        
        # Check if model has predict_proba (like your example)
        if hasattr(model, "predict_proba"):
            prob, conf = model.predict_proba(inputs, return_confidence=True)
            return {
                "probability": prob,
                "confidence": conf,
                "model_type": "custom_service"
            }
        
        # Fallback: return model info
        return {"error": "Custom model has no recognized predict method", "model_type": "unknown"}
    
    def _build_schema(self, config: Dict) -> Dict:
        """Build tool schema for orchestrator from config."""
        properties = {}
        required = []
        
        for feature in config.get("features", []):
            feat_config = config.get("feature_info", {}).get(feature, {})
            feat_type = feat_config.get("type", "number")
            
            properties[feature] = {
                "type": feat_type,
                "description": feat_config.get("description", f"Input feature: {feature}")
            }
            
            if feat_config.get("required", True):
                required.append(feature)
        
        return {
            "name": config["id"],
            "description": config.get("description", f"ML model prediction for {config['id']}"),
            "input_schema": {
                "type": "object",
                "properties": properties,
                "required": required
            },
            "output_info": config.get("output", {"type": "prediction"})
        }
    
    def get_tool_schemas(self) -> List[Dict]:
        """Get all tool schemas for prompt building."""
        return list(self.model_schemas.values())
    
    def get_model(self, model_id: str) -> Optional[Dict]:
        """Get loaded model entry."""
        return self.loaded_models.get(model_id)


# --- ORCHESTRATOR WITH DYNAMIC MODEL LOADING ---

class MLOrchestrator:
    """Orchestrator that routes to ML models loaded from .pkl files."""
    
    def __init__(
        self,
        model_config_path: str,
        llm_base_model: str = "Qwen/Qwen2.5-1.5B-Instruct",
        llm_adapter_path: Optional[str] = None,
        load_in_4bit: bool = True
    ):
        # Load ML models from config
        self.registry = ModelRegistry(model_config_path)
        self.tools = self.registry.load_all_models()
        
        # Load LLM for orchestration
        self.tokenizer = AutoTokenizer.from_pretrained(llm_base_model, trust_remote_code=True)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        
        print(f"\n🤖 Loading LLM: {llm_base_model}")
        
        if load_in_4bit and torch.cuda.is_available():
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
            )
            self.model = AutoModelForCausalLM.from_pretrained(
                llm_base_model,
                quantization_config=bnb_config,
                device_map="auto",
                trust_remote_code=True
            )
        else:
            self.model = AutoModelForCausalLM.from_pretrained(
                llm_base_model,
                device_map="auto" if torch.cuda.is_available() else None,
                trust_remote_code=True
            )
        
        if llm_adapter_path and os.path.exists(llm_adapter_path):
            print(f"🔄 Loading adapter: {llm_adapter_path}")
            self.model = PeftModel.from_pretrained(self.model, llm_adapter_path)
        
        self.model.eval()
        print("✅ Orchestrator ready")
    
    def build_prompt(self, query: str) -> str:
        """Build prompt with available ML models as tools."""
        tools = self.registry.get_tool_schemas()
        
        system_msg = f"""You are an ML Model Orchestrator.
Analyze the user request and select the appropriate ML model(s) to make predictions.
Use the EXACT model ID and parameter names from the schemas.

Available ML Models:
{json.dumps(tools, indent=2)}

Respond with JSON:
{{
  "plan_type": "single_step",
  "tool_use": {{
    "tool_name": "<model_id>",
    "arguments": {{<feature_name>: <value>}}
  }}
}}"""
        
        messages = [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": query}
        ]
        
        return self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    def parse_output(self, text: str) -> Optional[Dict]:
        """Parse model selection."""
        # Extract JSON
        match = re.search(r'```(?:json)?\s*([\s\S]*?)\s*```', text)
        json_str = match.group(1) if match else re.search(r'(\{[\s\S]*\})', text)
        
        if not json_str:
            return None
        
        try:
            if isinstance(json_str, re.Match):
                json_str = json_str.group(1)
            data = json.loads(json_str)
            
            # Normalize format
            if "tool_use" in data:
                return data
            elif "tool_name" in data:
                return {
                    "plan_type": data.get("plan_type", "single_step"),
                    "tool_use": {
                        "tool_name": data["tool_name"],
                        "arguments": data.get("arguments", data.get("kwargs", {}))
                    }
                }
            return None
        except:
            return None
    
    @torch.no_grad()
    def predict(self, query: str, temperature: float = 0.1) -> Dict[str, Any]:
        """
        End-to-end: Understand query -> Select model -> Run prediction.
        """
        prompt = self.build_prompt(query)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        
        outputs = self.model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )
        
        generated = self.tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        plan = self.parse_output(generated)
        
        if not plan:
            return {
                "success": False,
                "error": "Failed to parse orchestration plan",
                "raw_output": generated
            }
        
        # Execute ML model
        tool_name = plan["tool_use"]["tool_name"]
        arguments = plan["tool_use"]["arguments"]
        
        if tool_name not in self.tools:
            return {
                "success": False,
                "error": f"Model '{tool_name}' not found",
                "available": list(self.tools.keys()),
                "raw_output": generated
            }
        
        # Run the actual ML model
        tool_func = self.tools[tool_name]
        prediction_result = tool_func(**arguments)
        
        return {
            "success": "error" not in prediction_result,
            "query": query,
            "selected_model": tool_name,
            "arguments": arguments,
            "prediction": prediction_result,
            "raw_orchestration": generated
        }


# --- EXAMPLE CONFIG FILE (models_config.json) ---

EXAMPLE_CONFIG = {
    "models": [
        {
            "id": "donor_prediction",
            "description": "Predicts blood donor eligibility and donation probability based on donor history and demographics",
            "file_path": "/kaggle/input/online-logestic-regression/other/default/1/online_logistic_regression_model.pkl",
            "type": "custom",
            "features": [
                "recency_days",
                "donation_count_last_12m",
                "lifetime_donation_count",
                "age",
                "bmi",
                "blood_type",
                "sex",
                "is_regular_donor"
            ],
            "feature_info": {
                "recency_days": {"type": "integer", "description": "Days since last donation", "required": True},
                "donation_count_last_12m": {"type": "integer", "description": "Donations in last year", "required": True},
                "lifetime_donation_count": {"type": "integer", "description": "Total lifetime donations", "required": True},
                "age": {"type": "integer", "description": "Donor age in years", "required": True},
                "bmi": {"type": "number", "description": "Body mass index", "required": True},
                "blood_type": {"type": "string", "description": "Blood type (A+, O-, etc.)", "required": True},
                "sex": {"type": "string", "description": "M or F", "required": True},
                "is_regular_donor": {"type": "integer", "description": "1 if regular donor, 0 otherwise", "required": True}
            },
            "output": {
                "type": "probability",
                "description": "Probability of donation in next 6 months"
            }
        },
    ]
}


# --- USAGE EXAMPLE ---

def example_usage():
    """Demonstrate ML orchestration with .pkl models."""
    
    # First, save example config (in real use, you'd have this file)
    config_path = Path("ml_models/models_config.json")
    config_path.parent.mkdir(exist_ok=True)
    
    with open(config_path, 'w') as f:
        json.dump(EXAMPLE_CONFIG, f, indent=2)
    
    print("Example config saved. In production, place your .pkl files in the same directory.")
    
    # Initialize orchestrator
    orchestrator = MLOrchestrator(
        model_config_path=str(config_path),
        llm_base_model="Qwen/Qwen2.5-1.5B-Instruct",
        llm_adapter_path="/kaggle/working/qwen2.5-orchestrator-10k",  # Your fine-tuned adapter
        load_in_4bit=True
    )
    
    # Example queries
    test_queries = [
        "Check if this donor is likely to donate: 45 days since last donation, 3 donations last year, age 35, BMI 24, blood type O-, female, regular donor",
    ]
    
    print("\n" + "="*70)
    print("ML MODEL ORCHESTRATION TESTS")
    print("="*70)
    
    for query in test_queries:
        print(f"\n📝 Query: {query[:80]}...")
        result = orchestrator.predict(query, temperature=0.1)
        
        if result["success"]:
            print(f"🔧 Model: {result['selected_model']}")
            print(f"📥 Args: {result['arguments']}")
            print(f"🔮 Prediction: {result['prediction']}")
        else:
            print(f"❌ Error: {result.get('error')}")
            print(f"Raw: {result.get('raw_orchestration', 'N/A')[:200]}...")


if __name__ == "__main__":
    example_usage()

loading mdoels
model config {'id': 'donor_prediction', 'description': 'Predicts blood donor eligibility and donation probability based on donor history and demographics', 'file_path': '/kaggle/input/online-logestic-regression/other/default/1/online_logistic_regression_model.pkl', 'type': 'custom', 'features': ['recency_days', 'donation_count_last_12m', 'lifetime_donation_count', 'age', 'bmi', 'blood_type', 'sex', 'is_regular_donor'], 'feature_info': {'recency_days': {'type': 'integer', 'description': 'Days since last donation', 'required': True}, 'donation_count_last_12m': {'type': 'integer', 'description': 'Donations in last year', 'required': True}, 'lifetime_donation_count': {'type': 'integer', 'description': 'Total lifetime donations', 'required': True}, 'age': {'type': 'integer', 'description': 'Donor age in years', 'required': True}, 'bmi': {'type': 'number', 'description': 'Body mass index', 'required': True}, 'blood_type': {'type': 'string', 'description': 'Blood type (A+, O-, 

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

🔄 Loading adapter: /kaggle/working/qwen2.5-orchestrator-10k
✅ Orchestrator ready
{'tools': {'donor_prediction': <function ModelRegistry._create_tool_wrapper.<locals>.tool_function at 0x7ee2f027b740>}}

ML MODEL ORCHESTRATION TESTS

📝 Query: Check if this donor is likely to donate: 45 days since last donation, 3 donation...
tools when building prompt: [
  {
    "name": "donor_prediction",
    "description": "Predicts blood donor eligibility and donation probability based on donor history and demographics",
    "input_schema": {
      "type": "object",
      "properties": {
        "recency_days": {
          "type": "integer",
          "description": "Days since last donation"
        },
        "donation_count_last_12m": {
          "type": "integer",
          "description": "Donations in last year"
        },
        "lifetime_donation_count": {
          "type": "integer",
          "description": "Total lifetime donations"
        },
        "age": {
          "type": "integer",
 

In [6]:
import os
import json
import pickle
import re
import warnings
import time
from pathlib import Path
from typing import List, Dict, Any, Optional

import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Suppress warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. UNIVERSAL INFERENCE ENGINE
# ==========================================
class UniversalInferenceEngine:
    def __init__(self, model_path: str):
        self.model_path = Path(model_path)
        self.model_data = self._load_model(self.model_path)
        self.model_type = self._detect_type(self.model_data)

    def _load_model(self, path: Path):
        if not path.exists():
            raise FileNotFoundError(f"Model file not found: {path}")
        with open(path, 'rb') as f:
            return pickle.load(f)

    def _detect_type(self, data: Any) -> str:
        if isinstance(data, dict) and "architecture" in data:
            return data["architecture"] 
        if hasattr(data, "predict"):
            return "sklearn"
        return "unknown"

    def predict(self, inputs: Dict[str, Any]) -> Dict[str, Any]:
        try:
            if self.model_type == "linear_custom":
                return self._predict_custom_linear(inputs)
            elif self.model_type == "sklearn":
                return self._predict_sklearn(inputs)
            else:
                return {"error": f"Unsupported model type: {self.model_type}"}
        except Exception as e:
            return {"error": str(e)}

    def _predict_custom_linear(self, inputs: Dict):
        params = self.model_data["params"]
        means = params.get("means", {})
        stds = params.get("stds", {})
        encoders = params.get("encoders", {})
        weights = params.get("weights", {})
        intercept = params.get("intercept", 0.0)

        z = intercept
        for feat, val in inputs.items():
            if feat not in weights: continue
            if isinstance(val, str) and feat in encoders:
                classes = encoders[feat]
                val = float(classes.index(val)) if val in classes else 0.0
            else:
                val = float(val)
            if feat in means:
                m, s = means[feat], stds[feat]
                val = (val - m) / s if s > 0 else 0.0
                val = max(min(val, 5.0), -5.0)
            z += val * weights[feat]

        prob = 1.0 / (1.0 + np.exp(-max(min(z, 25), -25)))
        return {"probability": float(prob), "logit": float(z)}

    def _predict_sklearn(self, inputs: Dict):
        model = self.model_data.get("model_binary", self.model_data)
        features = self.model_data.get("metadata", {}).get("features", [])
        if features:
            input_vector = [inputs.get(f, 0) for f in features]
            X = np.array([input_vector])
        else:
            X = np.array([list(inputs.values())])
        pred = model.predict(X)[0]
        result = {"prediction": pred}
        if hasattr(model, "predict_proba"):
            result["probability"] = model.predict_proba(X)[0].tolist()
        return result

# ==========================================
# 2. MODEL REGISTRY
# ==========================================
class ModelRegistry:
    def __init__(self, config_path: str):
        self.config_path = Path(config_path)
        self.models_dir = self.config_path.parent
        self.config = self._load_config()
        self.model_metadata: Dict[str, Dict] = {} 
        
    def _load_config(self) -> Dict:
        with open(self.config_path, 'r') as f:
            return json.load(f)
    
    def load_all_models(self) -> Dict[str, Any]:
        tools = {}
        for model_config in self.config.get("models", []):
            model_id = model_config["id"]
            try:
                file_path = self.models_dir / model_config["file_path"]
                engine = UniversalInferenceEngine(str(file_path))
                def tool_function(**kwargs):
                    return engine.predict(kwargs)
                tools[model_id] = tool_function
                self.model_metadata[model_id] = {
                    "schema": self._build_schema(model_config),
                    "examples": model_config.get("examples", [])
                }
                print(f"✅ Registered: {model_id}")
            except Exception as e:
                print(f"❌ Failed {model_id}: {e}")
        return tools

    def _build_schema(self, config: Dict) -> Dict:
        properties = {}
        required = []
        for feature in config.get("features", []):
            feat_info = config.get("feature_info", {}).get(feature, {})
            properties[feature] = {
                "type": feat_info.get("type", "number"),
                "description": feat_info.get("description", f"Input feature: {feature}")
            }
            required.append(feature)
        return {
            "name": config["id"],
            "description": config.get("description", ""),
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": required
            }
        }

    def get_all_metadata(self) -> List[Dict]:
        return list(self.model_metadata.values())

# ==========================================
# 3. LLM ORCHESTRATOR
# ==========================================
class MLOrchestrator:
    def __init__(self, model_config_path: str, llm_base_model: str):
        self.registry = ModelRegistry(model_config_path)
        self.tools = self.registry.load_all_models()
        
        print(f"🤖 Loading LLM: {llm_base_model}")
        self.tokenizer = AutoTokenizer.from_pretrained(llm_base_model, trust_remote_code=True)
        
        self.device = "cpu"
        quantization_config = None
        torch_dtype = torch.float32

        if torch.cuda.is_available():
            print("🚀 Using CUDA (Nvidia) with 4-bit quantization")
            self.device = "cuda"
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16
            )
            torch_dtype = torch.float16
        elif torch.backends.mps.is_available():
            print("🍎 Using MPS (Apple Silicon) - Float16 mode")
            self.device = "mps"
            torch_dtype = torch.float16
        else:
            print("⚠️ Using CPU (Slow)")
        
        self.model = AutoModelForCausalLM.from_pretrained(
            llm_base_model,
            quantization_config=quantization_config,
            torch_dtype=torch_dtype,
            device_map="auto" if self.device == "cuda" else None,
            trust_remote_code=True
        )
        
        if self.device == "mps":
            self.model.to("mps")

    def build_router_prompt(self, query: str) -> str:
        metadata_list = self.registry.get_all_metadata()
        tools_def = [m["schema"] for m in metadata_list]
        
        few_shot = ""
        for meta in metadata_list:
            model_name = meta["schema"]["name"]
            for ex in meta["examples"]:
                js = {"tool_use": {"tool_name": model_name, "arguments": ex["extracted_args"]}}
                # Minimal tokens for few-shot to save context
                few_shot += f"User: {ex['user_query']}\nAssistant: ```json\n{json.dumps(js)}\n```\n"

        return f"""You are an API Router. Extract parameters from the query into JSON.
Available Tools:
{json.dumps(tools_def, indent=2)}

Rules:
1. Output ONLY JSON.
2. If values seem extreme, extract them anyway.

Examples:
{few_shot}
User: {query}
Assistant:"""

    def parse_output(self, text: str) -> Optional[Dict]:
        """
        Ultra-Robust Parser for Small Models.
        Ignores markdown, finds the first {, and counts braces to find the end.
        Handles the ']' ending error common in 0.5B models.
        """
        # Remove markdown wrappers immediately
        text = text.replace("```json", "").replace("```", "").strip()

        start_index = text.find("{")
        if start_index == -1:
            return None

        # Stack-based extraction
        balance = 0
        end_index = -1
        
        # Iterate starting from the first {
        for i, char in enumerate(text[start_index:], start=start_index):
            if char == "{":
                balance += 1
            elif char == "}":
                balance -= 1
            elif char == "]" and balance == 1: 
                # CRITICAL FIX for 0.5B models: 
                # If model ends with ']' instead of '}' for the main object, treat it as '}'
                balance -= 1
                text = text[:i] + "}" + text[i+1:] # patch string

            if balance == 0:
                end_index = i + 1
                break

        if end_index == -1:
            # If we didn't find a clean close, try to parse what we have up to the last char
            candidate = text[start_index:]
        else:
            candidate = text[start_index:end_index]

        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            # Last ditch effort: Try to fix single quotes
            try:
                return json.loads(candidate.replace("'", '"'))
            except:
                return None

    @torch.no_grad()
    def predict(self, query: str) -> Dict:
        # --- PHASE 1: ROUTING ---
        router_prompt = self.build_router_prompt(query)
        inputs = self.tokenizer(router_prompt, return_tensors="pt").to(self.device)
        
        # Reduced max_tokens significantly to speed up and prevent rambling
        outputs = self.model.generate(
            **inputs, 
            max_new_tokens=80, 
            temperature=0.01, 
            do_sample=False,
            pad_token_id=self.tokenizer.eos_token_id,
            repetition_penalty=1.2 # Stops loops
        )
        generated_router = self.tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        
        plan = self.parse_output(generated_router)
        
        if not plan or "tool_use" not in plan:
             return {"success": False, "error": "Routing failed", "raw": generated_router}
             
        tool = plan["tool_use"]["tool_name"]
        args = plan["tool_use"]["arguments"]
        
        if tool not in self.tools:
            return {"success": False, "error": "Tool not found"}
            
        ml_result = self.tools[tool](**args)
        raw_prob = ml_result.get('probability', 0.0)
        
        # --- PHASE 2: EXPLANATION ---
        # Very short prompt for speed
        explain_prompt = f"""User: "{query}"
Prediction: {raw_prob:.2f} (0=No, 1=Yes).
Task: Answer in user's language. Keep it very short.

Response:"""
        
        inputs_exp = self.tokenizer(explain_prompt, return_tensors="pt").to(self.device)
        outputs_exp = self.model.generate(
            **inputs_exp, max_new_tokens=60, temperature=0.3, do_sample=True,
            pad_token_id=self.tokenizer.eos_token_id
        )
        explanation = self.tokenizer.decode(outputs_exp[0][inputs_exp['input_ids'].shape[1]:], skip_special_tokens=True)
        
        # Cleanup
        if "Response:" in explanation:
            explanation = explanation.split("Response:")[-1].strip()

        return {
            "success": True,
            "inputs": args,
            "prediction_raw": ml_result,
            "natural_language_response": explanation
        }

# ==========================================
# 4. RUNNER
# ==========================================

def main():
    # SETUP CONFIG & DUMMY FILES
    Path("ml_models").mkdir(exist_ok=True)
    Path("models").mkdir(exist_ok=True)
    
    # Dummy Model for testing
    model_path = Path("/Users/mac/Documents/pios-1/ml-backend/models/donor_v1.pkl")
    if not model_path.exists():
        dummy_data = {
            "architecture": "linear_custom", 
            "params": {"weights": {"age": -0.01}, "intercept": 0.5}
        }
        with open(model_path, "wb") as f:
            pickle.dump(dummy_data, f)

    config = {
      "models": [
        {
          "id": "donor_prediction",
          "description": "Predicts blood donor eligibility.",
          "file_path": "/Users/mac/Documents/pios-1/ml-backend/models/donor_v1.pkl", 
          "features": ["recency_days", "donation_count_last_12m", "age", "bmi", "sex"],
          "feature_info": { "age": {"type": "integer"}, "sex": {"type": "string"} },
          "examples": [
            {
              "user_query": "Check 35 year old Male",
              "extracted_args": { "age": 35, "sex": "M", "bmi": 24, "recency_days": 45, "donation_count_last_12m": 3 }
            }
          ]
        },
        {
          "id": "xgb_demand_forecast_j+30",
          "description": "Predicts blood demand forecast for 30 days ahead.",
          "file_path": "/Users/mac/Documents/pios-1/ml-backend/models/xgb_demand_forecast_j+30.pkl", 
          "features": ['dow', 'weekend', 'month', 'holiday', 'temp_c', 'rain_mm', 'flu_index', 'trauma_cases', 'scheduled_surgeries', 'donation_campaign', 'supply_shock', 'stock_start', 'units_collected', 'wastage', 'lag1', 'lag7', 'lag30', 'roll7', 'roll30', 'hospital_East', 'hospital_North', 'hospital_West', 'blood_type_A-', 'blood_type_AB+', 'blood_type_AB-', 'blood_type_B+', 'blood_type_B-', 'blood_type_O+', 'blood_type_O-'],
           "feature_info": {
                "dow": {"type": "integer"},
                "weekend": {"type": "integer"},
                "month": {"type": "integer"},
                "holiday": {"type": "integer"},
                "temp_c": {"type": "float"},
                "rain_mm": {"type": "float"},
                "flu_index": {"type": "float"},
                "trauma_cases": {"type": "float"},
                "scheduled_surgeries": {"type": "float"},
                "donation_campaign": {"type": "integer"},
                "supply_shock": {"type": "integer"},
                "stock_start": {"type": "float"},
                "units_collected": {"type": "float"},
                "wastage": {"type": "float"},
                "lag1": {"type": "float"},
                "lag7": {"type": "float"},
                "lag30": {"type": "float"},
                "roll7": {"type": "float"},
                "roll30": {"type": "float"},
                "hospital_East": {"type": "float"},
                "hospital_North": {"type": "float"},
                "hospital_West": {"type": "float"},
                "blood_type_A-": {"type": "float"},
                "blood_type_AB+": {"type": "float"},
                "blood_type_AB-": {"type": "float"},
                "blood_type_B+": {"type": "float"},
                "blood_type_B-": {"type": "float"},
                "blood_type_O+": {"type": "float"},
                "blood_type_O-": {"type": "float"}
            },
          "examples": [
            {
              "user_query": "Predict demand for hospital East, blood type A- for 30 days ahead.",
              "extracted_args": {
                "hospital": "East", "blood_type": "A-", "days_ahead": 30
              }
            }
          ]
        }
      ]
    }
    
    with open("ml_models/config.json", "w") as f:
        json.dump(config, f, indent=2)

    try:
        start_time = time.time()
        orchestrator = MLOrchestrator(
            model_config_path="ml_models/config.json",
            llm_base_model="Qwen/Qwen2.5-1.5B-Instruct"
        )
        load_time = time.time() - start_time
        print(f"⏱️ Model Load Time: {load_time:.2f}s")
        
        queries = [
            "Can you check if a 35 year old Male with BMI 24 who last donated 45 days ago (3 times this year) is likely to donate again?",
            # "can a 90 year old Male with BMI 10 who last donated 45 days ago (0 times this year) donate again?",
            # "Un homme de 90 ans avec un IMC de 10 qui a fait un dernier don il y a 45 jours (0 fois cette année) peut-il faire un nouveau donor ?"
            # "Predict demand for hospital East, blood type A- for 30 days ahead."
        ]

        print("\n" + "="*60)
        for q in queries:
            q_start = time.time()
            print(f"\n📝 Query: {q}")
            res = orchestrator.predict(q)
            
            if res.get('success'):
                prob = res['prediction_raw'].get('probability', 0)
                print(f"📊 ML Output: {prob:.4f}")
                print(f"🗣️ Response: {res['natural_language_response']}")
            else:
                print(f"❌ Error: {res.get('error')}")
                print(f"   Raw: {res.get('raw', '')}")
            print(f"⏱️ Query Time: {time.time() - q_start:.2f}s")
                
    except Exception as e:
        print(f"CRITICAL ERROR: {e}")

if __name__ == "__main__":
    main()

✅ Registered: donor_prediction
✅ Registered: xgb_demand_forecast_j+30
🤖 Loading LLM: Qwen/Qwen2.5-1.5B-Instruct
🍎 Using MPS (Apple Silicon) - Float16 mode
⏱️ Model Load Time: 157.56s


📝 Query: Can you check if a 35 year old Male with BMI 24 who last donated 45 days ago (3 times this year) is likely to donate again?
📊 ML Output: 0.0000
🗣️ Response:  Yes, he might be eligible for another donation.
Explanation: The person meets the criteria of being over 35 years old, male, and has a normal BMI. They have also had their blood tested within the past 45 days, which means they haven't been sick recently. This suggests
⏱️ Query Time: 134.34s


In [4]:
import os
import json
import pickle
import re
import warnings
from pathlib import Path
from typing import List, Dict, Any, Optional, Callable, Union

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Suppress warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. UNIVERSAL INFERENCE ENGINE (Unchanged)
# ==========================================
class UniversalInferenceEngine:
    def __init__(self, model_path: str):
        self.model_path = Path(model_path)
        self.model_data = self._load_model(self.model_path)
        self.model_type = self._detect_type(self.model_data)

    def _load_model(self, path: Path):
        if not path.exists():
            raise FileNotFoundError(f"Model file not found: {path}")
        with open(path, 'rb') as f:
            return pickle.load(f)

    def _detect_type(self, data: Any) -> str:
        if isinstance(data, dict) and "architecture" in data:
            return data["architecture"] 
        if hasattr(data, "predict"):
            return "sklearn"
        if isinstance(data, torch.nn.Module) or (isinstance(data, dict) and "state_dict" in data):
            return "pytorch"
        return "unknown"

    def predict(self, inputs: Dict[str, Any]) -> Dict[str, Any]:
        try:
            if self.model_type == "linear_custom":
                return self._predict_custom_linear(inputs)
            elif self.model_type == "sklearn":
                return self._predict_sklearn(inputs)
            elif self.model_type == "pytorch" or self.model_type == "xgboost":
                return self._predict_pytorch(inputs)
            else:
                return {"error": f"Unsupported model type: {self.model_type}"}
        except Exception as e:
            return {"error": str(e), "model_type": self.model_type}

    def _predict_custom_linear(self, inputs: Dict):
        params = self.model_data["params"]
        means = params.get("means", {})
        stds = params.get("stds", {})
        encoders = params.get("encoders", {})
        weights = params.get("weights", {})
        intercept = params.get("intercept", 0.0)

        z = intercept
        for feat, val in inputs.items():
            if feat not in weights: continue
            if isinstance(val, str) and feat in encoders:
                classes = encoders[feat]
                val = float(classes.index(val)) if val in classes else 0.0
            else:
                val = float(val)
            if feat in means:
                m, s = means[feat], stds[feat]
                val = (val - m) / s if s > 0 else 0.0
                val = max(min(val, 5.0), -5.0)
            z += val * weights[feat]

        prob = 1.0 / (1.0 + np.exp(-max(min(z, 25), -25)))
        return {"probability": float(prob), "logit": float(z)}

    def _predict_sklearn(self, inputs: Dict):
        if isinstance(self.model_data, dict) and "model_binary" in self.model_data:
            model = self.model_data["model_binary"]
            features = self.model_data.get("metadata", {}).get("features", [])
        else:
            model = self.model_data
            features = []
        if features:
            input_vector = [inputs.get(f, 0) for f in features]
            X = np.array([input_vector])
        else:
            X = np.array([list(inputs.values())])
        pred = model.predict(X)[0]
        result = {"prediction": pred}
        if hasattr(model, "predict_proba"):
            result["probability"] = model.predict_proba(X)[0].tolist()
        return result

    def _predict_pytorch(self, inputs: Dict):
        return {"error": "Not implemented in this snippet"}

# ==========================================
# 2. MODEL REGISTRY (Unchanged)
# ==========================================
class ModelRegistry:
    def __init__(self, config_path: str):
        self.config_path = Path(config_path)
        self.models_dir = self.config_path.parent
        self.config = self._load_config()
        self.model_metadata: Dict[str, Dict] = {} 
        
    def _load_config(self) -> Dict:
        with open(self.config_path, 'r') as f:
            return json.load(f)
    
    def load_all_models(self) -> Dict[str, Any]:
        tools = {}
        for model_config in self.config.get("models", []):
            model_id = model_config["id"]
            try:
                file_path = self.models_dir / model_config["file_path"]
                engine = UniversalInferenceEngine(str(file_path))
                def tool_function(**kwargs):
                    return engine.predict(kwargs)
                tools[model_id] = tool_function
                self.model_metadata[model_id] = {
                    "schema": self._build_schema(model_config),
                    "examples": model_config.get("examples", [])
                }
                print(f"✅ Registered: {model_id}")
            except Exception as e:
                print(f"❌ Failed {model_id}: {e}")
        return tools

    def _build_schema(self, config: Dict) -> Dict:
        properties = {}
        required = []
        for feature in config.get("features", []):
            feat_info = config.get("feature_info", {}).get(feature, {})
            properties[feature] = {
                "type": feat_info.get("type", "number"),
                "description": feat_info.get("description", f"Input feature: {feature}")
            }
            required.append(feature)
        return {
            "name": config["id"],
            "description": config.get("description", ""),
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": required
            }
        }

    def get_all_metadata(self) -> List[Dict]:
        return list(self.model_metadata.values())

# ==========================================
# 3. LLM ORCHESTRATOR (Improved Explanation)
# ==========================================
class MLOrchestrator:
    def __init__(self, model_config_path: str, llm_base_model: str, adapter_path: str = None, load_in_4bit: bool = True):
        self.registry = ModelRegistry(model_config_path)
        self.tools = self.registry.load_all_models()
        
        print(f"🤖 Loading LLM Base: {llm_base_model}")
        self.tokenizer = AutoTokenizer.from_pretrained(llm_base_model, trust_remote_code=True)
        
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16
        ) if load_in_4bit and torch.cuda.is_available() else None
        
        # 1. Load the Base Model
        self.model = AutoModelForCausalLM.from_pretrained(
            llm_base_model, quantization_config=bnb_config, device_map="auto", trust_remote_code=True
        )

        # 2. LOAD THE FINE-TUNED ADAPTER (The Fix)
        if adapter_path:
            print(f"🔧 Loading LoRA Adapters from: {adapter_path}")
            # This attaches your fine-tuned weights to the base model
            self.model = PeftModel.from_pretrained(self.model, adapter_path)
        else:
            print("⚠️ No adapter path provided. Running raw base model.")


     
    def build_router_prompt(self, query: str) -> str:
            """Step 1: Extract JSON (Matched to Training Data)"""
            metadata_list = self.registry.get_all_metadata()
            # Get the schemas
            tools_def = [m["schema"] for m in metadata_list]
        
        # --- CRITICAL FIX: EXACT MATCH OF TRAINING PROMPT ---
        # The model was trained on this specific wording.
        # Do not add few-shot examples here if you didn't train with them inside the prompt context.
            system_msg = f"""You are an intelligent orchestrator.
            Select the correct tool based on the description and user query.
            Use the EXACT tool name and argument keys from the schema provided.

            TOOLS CONFIGURATION:
            {json.dumps(tools_def, indent=2)}"""

        # Format: System + User
        # We wrap it manually or rely on the tokenizer's chat template.
        # Since you used `dataset_text_field="messages"` in TRL, 
        # the model expects the chat template format.
        
            messages = [
                {"role": "system", "content": system_msg},
                {"role": "user", "content": query}
            ]
        
        # Apply the chat template (this handles <|im_start|>, <|im_end|> etc.)
            return self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    def build_explanation_prompt(self, query: str, prob_val: float) -> str:
        """Step 2: Natural Language (Multi-lingual)"""
        
        # We enforce language matching by giving examples in Eng/Fr/Ar
        # We also enforce concise answers to stop the model from rambling
        
        prompt = f"""You are a Donor Evaluation Assistant.
        Your goal: Interpret the AI probability score for the user in their own language.

        Rules:
        1. If Probability > 0.7: Encouraging "Yes".
        2. If Probability < 0.3: Polite "No".
        3. If 0.3 - 0.7: Neutral "Unsure/Borderline".
        4. OUTPUT MUST BE IN THE SAME LANGUAGE AS THE USER QUERY.
        5. Do not output "Answer:" or "Question:". Just the sentence.

        Examples:
        Query: "Can I donate?" (Prob: 0.95)
        Response: Yes, you are a great candidate! The model predicts a 95% chance of eligibility.

        Query: "Puis-je donner?" (Prob: 0.10)
        Response: Il semble peu probable que vous puissiez donner pour le moment (10% de probabilité).

        Query: "هل يمكنني التبرع؟" (Prob: 0.85)
        Response: نعم، أنت مرشح ممتاز! يتوقع النموذج احتمالية 85% للأهلية.

        Current Task:
        Query: "{query}" (Prob: {prob_val:.2f})
        Response:"""
        return prompt

    def parse_output(self, text: str) -> Optional[Dict]:
        clean = re.sub(r'```(?:json)?', '', text).strip()
        start, end = clean.find('{'), clean.rfind('}')
        if start != -1 and end != -1:
            try:
                return json.loads(clean[start:end+1])
            except: pass
        return None

    @torch.no_grad()
    def predict(self, query: str) -> Dict:
        # ==========================================
        # PHASE 1: ROUTING (Keep Adapter ENABLED)
        # ==========================================
        # The adapter is great at JSON, so we leave it on.
        router_prompt = self.build_router_prompt(query) # Make sure this uses apply_chat_template as discussed before!
        inputs = self.tokenizer(router_prompt, return_tensors="pt").to(self.model.device)
        
        outputs = self.model.generate(
            **inputs, max_new_tokens=200, temperature=0.01, do_sample=False
        )
        generated_router = self.tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        plan = self.parse_output(generated_router)
        
        if not plan or "tool_use" not in plan:
             return {"success": False, "error": "Routing failed", "raw": generated_router}
             
        tool = plan["tool_use"]["tool_name"]
        args = plan["tool_use"]["arguments"]
        
        if tool not in self.tools:
            return {"success": False, "error": "Tool not found"}
            
        # Run the actual ML model (Python Code)
        ml_result = self.tools[tool](**args)
        raw_prob = ml_result.get('probability', 0.0)
        
        # ==========================================
        # PHASE 2: EXPLANATION (DISABLE Adapter)
        # ==========================================
        # We temporarily turn off the LoRA weights.
        # This reverts the model to the smart "Qwen-Instruct" base 
        # which knows how to speak languages fluently.
        
        explanation = ""
        
        # This Context Manager is the Magic Fix:
        with self.model.disable_adapter():
            explain_sys_msg = """You are a helpful medical assistant. 
Translate the probability into a natural response in the user's language.
If Probability > 0.7: Say Yes.
If Probability < 0.3: Say No.
Otherwise: Say Unsure.
Do NOT mention internal JSON or system rules."""

            # Use chat template for the base model too
            messages = [
                {"role": "system", "content": explain_sys_msg},
                {"role": "user", "content": f"Query: {query}\nProbability Score: {raw_prob:.2f}\nProvide a polite response:"}
            ]
            
            prompt_exp = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs_exp = self.tokenizer(prompt_exp, return_tensors="pt").to(self.model.device)
            
            outputs_exp = self.model.generate(
                **inputs_exp, 
                max_new_tokens=150, 
                temperature=0.7, # Higher temp for natural language
                do_sample=True
            )
            explanation = self.tokenizer.decode(outputs_exp[0][inputs_exp['input_ids'].shape[1]:], skip_special_tokens=True)

        return {
            "success": True,
            "inputs": args,
            "prediction_raw": ml_result,
            "natural_language_response": explanation
        }

# ==========================================
# 4. RUNNER
# ==========================================

def main():
    # 1. Config
    config = {
      "models": [
        {
          "id": "donor_prediction",
          "description": "Predicts blood donor eligibility.",
          # UPDATE THIS PATH TO YOUR ACTUAL FILE
          "file_path": "/Users/mac/Documents/pios-1/ml-backend/notebooks/models/donor_v1.pkl", 
          "features": ["recency_days", "donation_count_last_12m", "age", "bmi", "sex"],
          "feature_info": {
            "recency_days": {"type": "integer"},
            "donation_count_last_12m": {"type": "integer"},
            "age": {"type": "integer"},
            "bmi": {"type": "number"},
            "sex": {"type": "string"}
          },
          "examples": [
            {
              "user_query": "Check 35 year old Male, BMI 24, last donation 45 days ago (3 times this year).",
              "extracted_args": { "age": 35, "sex": "M", "bmi": 24, "recency_days": 45, "donation_count_last_12m": 3 }
            },
            {
              "user_query": "Can a 90 year old male with BMI 10 donate? Last time 100 days ago, 0 times this year.",
              "extracted_args": { "age": 90, "sex": "M", "bmi": 10, "recency_days": 100, "donation_count_last_12m": 0 }
            }
          ]
        },
           {
          "id": "xgb_demand_forecast_j+30",
          "description": "Predicts blood demand forecast for 30 days ahead.",
          "file_path": "/Users/mac/Documents/pios-1/ml-backend/notebooks/models/xgb_demand_forecast_j+30.pkl", 
          "features": ['dow', 'weekend', 'month', 'holiday', 'temp_c', 'rain_mm', 'flu_index', 'trauma_cases', 'scheduled_surgeries', 'donation_campaign', 'supply_shock', 'stock_start', 'units_collected', 'wastage', 'lag1', 'lag7', 'lag30', 'roll7', 'roll30', 'hospital_East', 'hospital_North', 'hospital_West', 'blood_type_A-', 'blood_type_AB+', 'blood_type_AB-', 'blood_type_B+', 'blood_type_B-', 'blood_type_O+', 'blood_type_O-'],
           "feature_info": {
                "dow": {"type": "integer"},
                "weekend": {"type": "integer"},
                "month": {"type": "integer"},
                "holiday": {"type": "integer"},
                "temp_c": {"type": "float"},
                "rain_mm": {"type": "float"},
                "flu_index": {"type": "float"},
                "trauma_cases": {"type": "float"},
                "scheduled_surgeries": {"type": "float"},
                "donation_campaign": {"type": "integer"},
                "supply_shock": {"type": "integer"},
                "stock_start": {"type": "float"},
                "units_collected": {"type": "float"},
                "wastage": {"type": "float"},
                "lag1": {"type": "float"},
                "lag7": {"type": "float"},
                "lag30": {"type": "float"},
                "roll7": {"type": "float"},
                "roll30": {"type": "float"},
                "hospital_East": {"type": "float"},
                "hospital_North": {"type": "float"},
                "hospital_West": {"type": "float"},
                "blood_type_A-": {"type": "float"},
                "blood_type_AB+": {"type": "float"},
                "blood_type_AB-": {"type": "float"},
                "blood_type_B+": {"type": "float"},
                "blood_type_B-": {"type": "float"},
                "blood_type_O+": {"type": "float"},
                "blood_type_O-": {"type": "float"}
            },
          "examples": [
            {
                "user_query": "Baseline weekday forecast for O+ at East hospital",
                "extracted_args": {
                  "dow": 2,
                  "weekend": 0,
                  "month": 3,
                  "holiday": 0,
                  "temp_c": 18.5,
                  "rain_mm": 2.1,
                  "flu_index": 0.3,
                  "trauma_cases": 14,
                  "scheduled_surgeries": 22,
                  "donation_campaign": 0,
                  "supply_shock": 0,
                  "stock_start": 420,
                  "units_collected": 38,
                  "wastage": 4,
                  "lag1": 52,
                  "lag7": 48,
                  "lag30": 45,
                  "roll7": 50,
                  "roll30": 47,
                  "hospital_East": 1,
                  "hospital_North": 0,
                  "hospital_West": 0,
                  "blood_type_A-": 0,
                  "blood_type_AB+": 0,
                  "blood_type_AB-": 0,
                  "blood_type_B+": 0,
                  "blood_type_B-": 0,
                  "blood_type_O+": 1,
                  "blood_type_O-": 0
                }
                },
            {
                 "user_query": "Winter flu surge scenario",
                 "extracted_args": {
                   "dow": 4,
                   "weekend": 0,
                   "month": 1,
                   "holiday": 0,
                   "temp_c": 6.2,
                   "rain_mm": 12.4,
                   "flu_index": 0.85,
                   "trauma_cases": 19,
                   "scheduled_surgeries": 28,
                   "donation_campaign": 0,
                   "supply_shock": 0,
                   "stock_start": 390,
                   "units_collected": 31,
                   "wastage": 6,
                   "lag1": 61,
                   "lag7": 57,
                   "lag30": 54,
                   "roll7": 59,
                   "roll30": 55,
                   "hospital_East": 0,
                   "hospital_North": 1,
                   "hospital_West": 0,
                   "blood_type_A-": 0,
                   "blood_type_AB+": 0,
                   "blood_type_AB-": 0,
                   "blood_type_B+": 1,
                   "blood_type_B-": 0,
                   "blood_type_O+": 0,
                   "blood_type_O-": 0
                 }
            },
              {
                  "user_query": "Donation campaign after supply shock",
                  "extracted_args": {
                    "dow": 6,
                    "weekend": 1,
                    "month": 7,
                    "holiday": 0,
                    "temp_c": 31.8,
                    "rain_mm": 0.0,
                    "flu_index": 0.1,
                    "trauma_cases": 26,
                    "scheduled_surgeries": 18,
                    "donation_campaign": 1,
                    "supply_shock": 1,
                    "stock_start": 260,
                    "units_collected": 74,
                    "wastage": 3,
                    "lag1": 72,
                    "lag7": 66,
                    "lag30": 59,
                    "roll7": 68,
                    "roll30": 61,
                    "hospital_East": 0,
                    "hospital_North": 0,
                    "hospital_West": 1,
                    "blood_type_A-": 0,
                    "blood_type_AB+": 1,
                    "blood_type_AB-": 0,
                    "blood_type_B+": 0,
                    "blood_type_B-": 0,
                    "blood_type_O+": 0,
                    "blood_type_O-": 0
                  }
                }
          ]
}]
}
    
    config_path = Path("ml_models/config.json")
    config_path.parent.mkdir(exist_ok=True, parents=True)
    with open(config_path, "w") as f:
        json.dump(config, f, indent=2)
    my_adapter_path = "/kaggle/working/qwen2.5-orchestrator-10k" 
    try:
        orchestrator = MLOrchestrator(
            model_config_path=str(config_path),
            llm_base_model="Qwen/Qwen2.5-1.5B-Instruct",
            adapter_path=my_adapter_path,
            load_in_4bit=True
        )
        
        queries = [
            # "Can you check if a 35 year old Male with BMI 24 who last donated 45 days ago (3 times this year) is likely to donate again?",
            # "can a 90 year old Male with BMI 10 who last donated 45 days ago (0 times this year) donate again?",
            # "Un homme de 35 ans (IMC 24) a donné il y a 45 jours (3 fois cette année). Est-il probable qu'il donne encore ?",
            # "هل يمكن لرجل يبلغ من العمر 90 عامًا ومؤشر كتلة جسمه 10 التبرع مرة أخرى؟"
            "Predict O-negative demand for the North hospital next month assuming winter flu is high and surgeries increase.",
            "What will blood demand look like after a donation campaign following a supply shortage?",
            "Forecast AB+ demand for West hospital on a hot summer weekend with no rain."
        ]

        print("\n" + "="*60)
        for q in queries:
            print(f"\n📝 Query: {q}")
            res = orchestrator.predict(q)
            
            if res.get('success'):
                prob = res['prediction_raw'].get('probability', 0)
                print(f"📊 ML Output: {prob:.4f}")
                print(f"🗣️ Response: {res['natural_language_response']}")
            else:
                print(f"❌ Error: {res.get('error')}")
                print(f"   Raw: {res.get('raw', '')}")
                
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    main()

✅ Registered: donor_prediction
✅ Registered: xgb_demand_forecast_j+30
🤖 Loading LLM Base: Qwen/Qwen2.5-1.5B-Instruct


Some parameters are on the meta device because they were offloaded to the disk.


🔧 Loading LoRA Adapters from: /kaggle/working/qwen2.5-orchestrator-10k
Error: Can't find 'adapter_config.json' at '/kaggle/working/qwen2.5-orchestrator-10k'
